# LAB 08 — Feature Engineering, Feature Selection & Explainability

**Module:** Feature Engineering, Feature Selection & Explainability
**Session:** 08
**Duration:** 150 minutes (10-minute break included)
**Audience:** LAB02-LAB07'yi tamamlamış katılımcılar (başlangıç seviyesi ağırlıklı, 2 deneyimli katılımcı)

## LAB02-LAB07'den Devam

LAB04'te öğrendiğiniz preprocessing (`ColumnTransformer`, `Pipeline`, data leakage önleme),
LAB06'daki classification metodolojisi ve LAB07'deki XGBoost/benchmarking yaklaşımı bu LAB'ın
temelini oluşturuyor. Bu konular burada **yeniden uzun biçimde anlatılmıyor** — kısa
hatırlatmalarla kullanılıyor.

## Bu LAB'ın Ana Hedefi

Bu LAB'ın amacı sadece daha başarılı bir model geliştirmek değildir. Amaç:

- Kaliteli feature'lar oluşturmayı,
- Gereksiz/zararlı feature'ları belirlemeyi,
- **Data Leakage risklerini sistemli biçimde yönetmeyi**,
- Model tahminlerini (Explainable AI / SHAP ile) açıklamayı

öğrenmektir.

## Bu LAB'ın Sonunda Yapabilecekleriniz

1. Feature Engineering kavramını açıklayabileceksiniz.
2. Feature Engineering ve Feature Selection arasındaki farkı anlayabileceksiniz.
3. Prediction-Time Feature Availability değerlendirmesi yapabileceksiniz.
4. Target Leakage ve Temporal Leakage risklerini belirleyebileceksiniz.
5. Date-Time Features oluşturabileceksiniz.
6. Interaction Features geliştirebileceksiniz.
7. Domain Knowledge kullanarak yeni feature'lar üretebileceksiniz.
8. Filter, Wrapper ve Embedded Methods arasındaki farkı açıklayabileceksiniz.
9. Feature Selection yöntemlerini Pipeline içerisinde uygulayabileceksiniz.
10. Baseline, Engineered ve Selected feature set'lerini karşılaştırabileceksiniz.
11. Model-Based Feature Importance sonuçlarını yorumlayabileceksiniz.
12. Permutation Importance uygulayabileceksiniz.
13. SHAP ile Global ve Local Explainability gerçekleştirebileceksiniz.
14. Model açıklamalarının sınırlamalarını değerlendirebileceksiniz.
15. Feature Engineering kararlarını deneysel sonuçlarla gerekçelendirebileceksiniz.

## Kullanılan Veri Kümesi: DataCo SMART Supply Chain

Bu LAB, önceki LAB'lardan **farklı** bir veri kümesi kullanır: **DataCo SMART Supply Chain**
(Kaggle: `shashwatwork/dataco-smart-supply-chain-for-big-data-analysis`, orijinal kaynak:
Mendeley Data, DOI `10.17632/8gx2fvg2k6.5`).

**İş Problemi: Pre-Dispatch Late Delivery Risk Prediction** — bir siparişin, sevkiyatından önce
gecikme riski taşıyıp taşımadığını tahmin etmek.

> **Prediction-Time Varsayımı:** Sipariş bilgisi ve planlanan sevkiyat yöntemi bilinir durumdadır,
> ancak gerçek sevkiyat ve teslimat sonuçları henüz bilinmemektedir. **Bu varsayımın gerçek bir
> operasyonel ortamda ayrıca doğrulanması gerekir** — bu LAB'ın veri kümesi, bu varsayımı
> doğrulayacak gerçek zamanlı bir sistem sağlamaz.

> **Önemli — Veri Gizliliği:** Bu veri kümesi gerçek müşteri bilgileri içerir (isim, e-posta,
> adres vb.). Bu LAB'da bu alanlar **hiçbir zaman belleğe yüklenmez** (`src/data_loading.py`
> içindeki `load_dataco_supply_chain()`, bu kolonları `usecols` ile dosya okuma aşamasında
> hariç tutar) ve Notebook çıktılarında hiçbir hassas değer görüntülenmez.

> **Önemli — Gözlem Granülaritesi:** Veri kümesindeki her satır bir **order item**'dır (sipariş
> kalemi), bir **sipariş (order)** değildir — bir sipariş birden fazla order item içerebilir. Bu
> LAB, order-item-level classification yaklaşımını kullanır, ancak aynı siparişin farklı veri
> bölmelerine düşmemesi için **group-aware split** uygular (Section 2 Part C).


---


## Section 1 — Feature Engineering Fundamentals & Dataset
**Süre: 15 dakika**

### Learning Objective

Feature Engineering, Feature Selection ve Feature Extraction arasındaki farkı kavramsal olarak
anlamak; DataCo veri kümesini güvenli biçimde (gizlilik-korumalı) yüklemek ve genel yapısını
incelemek.

### Concept Explanation

- **Feature Engineering**: Mevcut ham veriden, modelin daha iyi öğrenebileceği **daha anlamlı
  model girdileri** üretmek (örn. bir tarihten "hafta sonu mu?" bilgisini çıkarmak).
- **Feature Selection**: Zaten var olan (ham veya mühendislik ürünü) feature'lar arasından, model
  için **yararlı bir alt kümeyi seçmek** — yeni bir feature üretmez, var olanları eler.
- **Feature Extraction**: Mevcut feature'ları **farklı, genellikle daha düşük boyutlu bir temsile**
  dönüştürmek. **PCA (Principal Component Analysis)**, bu kategorinin klasik bir örneğidir: çok
  sayıda korelasyonlu feature'ı, varyansı en çok açıklayan daha az sayıda "bileşene" indirger. Bu
  LAB kapsamında **PCA implementasyonu zorunlu değildir** — burada yalnızca kavramsal olarak
  tanıtılmaktadır.

Bu üç kavram **birbirinin yerine geçmez**: Feature Engineering yeni sinyal üretir, Feature
Selection mevcut sinyalden gereksiz olanı eler, Feature Extraction mevcut sinyali yeniden temsil
eder.

### Hands-On — Veriyi Güvenli Biçimde Yükleme


In [ ]:
# Same environment setup pattern as LAB02-LAB07: resolve the project root, add src/ to sys.path,
# then load the shared configuration.
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)


In [ ]:
from data_loading import load_config, load_dataco_supply_chain

config = load_config()
RANDOM_STATE = config["lab08"]["random_seed"]
TARGET = config["lab08"]["target"]
POSITIVE_CLASS = config["lab08"]["positive_class"]
BENCHMARK_THREADS = config["lab08"]["benchmark"]["threads"]

# load_dataco_supply_chain() excludes PII columns (name, email, password, street, zip codes,
# customer/order-customer identifiers) at CSV read time via `usecols` -- they are never loaded
# into memory in the first place.
df = load_dataco_supply_chain(config)
print("Dataset shape (PII columns excluded at load time):", df.shape)
print("Target column:", TARGET, "| Positive class:", POSITIVE_CLASS)


### Hands-On — Genel Yapı


In [ ]:
print("Columns:")
print(df.columns.tolist())


In [ ]:
print("Data types summary:")
print(df.dtypes.value_counts())


In [ ]:
missing_summary = df.isna().sum()
missing_nonzero = missing_summary[missing_summary > 0]
print("Columns with missing values:")
print(missing_nonzero if len(missing_nonzero) else "None")


In [ ]:
numerical_candidates = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_candidates = df.select_dtypes(include=["object", "string"]).columns.tolist()
date_time_candidates = [c for c in df.columns if "date" in c.lower()]

print(f"Numerical columns ({len(numerical_candidates)}):", numerical_candidates)
print()
print(f"Categorical columns ({len(categorical_candidates)}):", categorical_candidates)
print()
print(f"Date-Time columns ({len(date_time_candidates)}):", date_time_candidates)


In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
target_pct = (df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2)
print("Target distribution (0 = No Late Delivery Risk per label, 1 = Late Delivery Risk):")
print(target_counts)
print()
print("Target distribution (%):")
print(target_pct)


In [ ]:
print("Unique Order Id count:", df["Order Id"].nunique())
print("Unique Order Item Id count:", df["Order Item Id"].nunique())
print("Total rows:", len(df))
print()
rows_per_order = df.groupby("Order Id").size()
print("Rows per Order Id -- describe():")
print(rows_per_order.describe())


### Output Interpretation

- Veri kümesi gerçekten **180,519 satır** içeriyor; `Order Item Id` benzersiz sayısı da
  **180,519** — yani her satır **tam olarak bir order item**'ı temsil ediyor (satır =
  order-item granülaritesi).
- `Order Id` benzersiz sayısı (**65,752**) satır sayısından **küçük** — yani bir sipariş
  birden fazla order item içerebiliyor (ortalama ~2.75 kayıt/sipariş, max 5). Bu, Section 2
  Part C'deki group-aware split kararının **gerçek verideki** gerekçesidir.
- Target dağılımı **orta derecede dengesiz** (yaklaşık %54.8 / %45.2) — LAB06/LAB07'deki E-Commerce
  Shipping verisine benzer bir oran, şiddetli bir imbalance değil.
- `Product Description` sütunu **tamamen boş** (tüm 180,519 satırda eksik) — bu, Section 2'de
  "Drop" kararı alınacak feature'lardan biri olacak.

### Student Question

**"Aynı Machine Learning algoritması farklı feature set'leriyle neden farklı performans
gösterebilir?"**

Bu soruyu şimdi tartışın — kesin cevabı yoktur. Section 5'te, **aynı XGBoost konfigürasyonunu**
üç farklı feature set'iyle (baseline/engineered/selected) eğitip gerçek sayılarla cevaplayacağız.


---


## Section 2 — Data Audit, Privacy & Target Leakage
**Süre: 20 dakika**

**Bu LAB'ın en kritik bölümlerinden biridir.**

### Learning Objective

Her feature için prediction-time availability, leakage riski ve PII riskini sistematik olarak
değerlendirmek; gerçek verilerle target leakage'ı somut biçimde göstermek; sipariş bütünlüğünü
koruyan kronolojik bir split kurmak.

### Part A — Feature Audit

Her feature'ın modele **varsayılan olarak dahil edilmeyeceğini**, aksine **kararın gerekçeli
biçimde verilmesi gerektiğini** kabul ediyoruz. Aşağıdaki tablo, `load_dataco_supply_chain()`'in
döndürdüğü 44 güvenli (PII-hariç) kolonun her biri için gerçek veri incelemesine dayanan bir
denetimdir.


In [ ]:
# Feature Audit -- built from real inspection of this dataset (duplicate-column checks,
# cardinality counts, missing-value counts), not from assumptions about what the columns "should"
# contain. See the Decision/Reason columns for the concrete evidence behind each call.
feature_audit_rows = [
    {"Feature": "Type", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "Payment type (DEBIT/TRANSFER/CASH/PAYMENT), chosen at order placement"},
    {"Feature": "Days for shipping (real)", "Feature_Type": "Numerical",
     "Prediction_Time_Available": "No", "Leakage_Risk": "High", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Actual shipping duration, known only after delivery -- direct outcome leakage"},
    {"Feature": "Days for shipment (scheduled)", "Feature_Type": "Numerical",
     "Prediction_Time_Available": "Yes (assumed)", "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "Planned shipping duration; assumed known at order time (training assumption)"},
    {"Feature": "Benefit per order", "Feature_Type": "Numerical", "Prediction_Time_Available": "Unverified",
     "Leakage_Risk": "Medium", "PII_Risk": "None", "Decision": "Review",
     "Reason": "Profit realization timing unverified; verified 100% identical to Order Profit Per Order"},
    {"Feature": "Sales per customer", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Review",
     "Reason": "Verified r=0.99 correlation with Sales / Order Item Total -- redundant"},
    {"Feature": "Delivery Status", "Feature_Type": "Categorical", "Prediction_Time_Available": "No",
     "Leakage_Risk": "CONFIRMED", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified deterministically identical to target (all 98,977 'Late delivery' rows == risk=1)"},
    {"Feature": TARGET, "Feature_Type": "Target", "Prediction_Time_Available": "N/A",
     "Leakage_Risk": "N/A", "PII_Risk": "None", "Decision": "Target",
     "Reason": "The label being predicted -- excluded from the feature matrix by definition"},
    {"Feature": "Category Id", "Feature_Type": "Numerical (id)", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "Verified 100% identical to Product Category Id -- keep one representation"},
    {"Feature": "Category Name", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "50 categories; human-readable product category, part of the baseline feature set"},
    {"Feature": "Customer City", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "Low", "Decision": "Review",
     "Reason": "563 unique values -- excluded from default baseline due to cardinality"},
    {"Feature": "Customer Country", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Review",
     "Reason": "Only 2 values, but redundant with Market/Order Region; not in default baseline"},
    {"Feature": "Customer Segment", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "3 categories; part of the baseline feature set"},
    {"Feature": "Customer State", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "Low", "Decision": "Review",
     "Reason": "46 unique values; redundant with Order Region/Market; not in default baseline"},
    {"Feature": "Department Id", "Feature_Type": "Numerical (id)", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "Redundant with Department Name -- keep one representation"},
    {"Feature": "Department Name", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "11 categories; part of the baseline feature set"},
    {"Feature": "Latitude", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "Medium", "Decision": "Drop",
     "Reason": "Near-continuous geocoordinate can approximate exact customer address -- privacy risk"},
    {"Feature": "Longitude", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "Medium", "Decision": "Drop",
     "Reason": "Same privacy concern as Latitude"},
    {"Feature": "Market", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "5 categories; part of the baseline feature set"},
    {"Feature": "Order City", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "Low", "Decision": "Review",
     "Reason": "3,597 unique values -- impractical for one-hot encoding in the default baseline"},
    {"Feature": "Order Country", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Review",
     "Reason": "164 unique values; redundant with Market/Order Region; not in default baseline"},
    {"Feature": "order date (DateOrders)", "Feature_Type": "Datetime", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Transform",
     "Reason": "Source for the group-aware split and for engineered Date-Time features (Section 3)"},
    {"Feature": "Order Id", "Feature_Type": "Identifier", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop (as feature)",
     "Reason": "Used only as the group key for the split; never used as a model feature"},
    {"Feature": "Order Item Cardprod Id", "Feature_Type": "Numerical (id)", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified 100% identical to Product Card Id; Category/Department already capture grouping"},
    {"Feature": "Order Item Discount", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Review",
     "Reason": "Potential redundancy with Order Item Discount Rate x Price"},
    {"Feature": "Order Item Discount Rate", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "Verified 0-1 fraction (max 0.25); part of the baseline feature set"},
    {"Feature": "Order Item Id", "Feature_Type": "Identifier", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Row-level unique identifier (1-to-1 with the row) -- no generalizable signal"},
    {"Feature": "Order Item Product Price", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "Part of the baseline feature set; verified 100% identical to Product Price"},
    {"Feature": "Order Item Profit Ratio", "Feature_Type": "Numerical", "Prediction_Time_Available": "Unverified",
     "Leakage_Risk": "Medium", "PII_Risk": "None", "Decision": "Review",
     "Reason": "Same profit-timing caution as Benefit per order"},
    {"Feature": "Order Item Quantity", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "Part of the baseline feature set"},
    {"Feature": "Sales", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Review",
     "Reason": "Verified r=0.99 with Order Item Total / Sales per customer -- redundant"},
    {"Feature": "Order Item Total", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Review",
     "Reason": "Redundant with Sales; overlaps with the engineered gross_item_value (Section 3)"},
    {"Feature": "Order Profit Per Order", "Feature_Type": "Numerical", "Prediction_Time_Available": "Unverified",
     "Leakage_Risk": "Medium", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified 100% identical to Benefit per order -- exact duplicate column"},
    {"Feature": "Order Region", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "23 categories; part of the baseline feature set"},
    {"Feature": "Order State", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "Low", "Decision": "Review",
     "Reason": "1,089 unique values -- extremely high cardinality, excluded from default baseline"},
    {"Feature": "Order Status", "Feature_Type": "Categorical", "Prediction_Time_Available": "Unverified",
     "Leakage_Risk": "Medium", "PII_Risk": "None", "Decision": "Review",
     "Reason": "CANCELED/SUSPECTED_FRAUD perfectly predict risk=0 (verified); ambiguous availability timing"},
    {"Feature": "Product Card Id", "Feature_Type": "Numerical (id)", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified 100% identical to Order Item Cardprod Id -- exact duplicate column"},
    {"Feature": "Product Category Id", "Feature_Type": "Numerical (id)", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified 100% identical to Category Id -- exact duplicate column"},
    {"Feature": "Product Description", "Feature_Type": "Text", "Prediction_Time_Available": "N/A",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified 100% missing (180,519 / 180,519 rows) -- unusable"},
    {"Feature": "Product Image", "Feature_Type": "Text (URL)", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Image URL string -- no predictive value"},
    {"Feature": "Product Name", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Review",
     "Reason": "118 unique values; redundant with Category/Department; optional for Advanced Challenge"},
    {"Feature": "Product Price", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified 100% identical to Order Item Product Price -- exact duplicate column"},
    {"Feature": "Product Status", "Feature_Type": "Numerical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Verified constant (single value 0 across all 180,519 rows) -- zero variance"},
    {"Feature": "shipping date (DateOrders)", "Feature_Type": "Datetime", "Prediction_Time_Available": "No",
     "Leakage_Risk": "High", "PII_Risk": "None", "Decision": "Drop",
     "Reason": "Actual shipment/delivery date, known only after shipping occurs -- outcome leakage"},
    {"Feature": "Shipping Mode", "Feature_Type": "Categorical", "Prediction_Time_Available": "Yes",
     "Leakage_Risk": "Low", "PII_Risk": "None", "Decision": "Keep",
     "Reason": "4 categories; part of the baseline feature set"},
]

feature_audit_df = pd.DataFrame(feature_audit_rows)
print("Feature Audit covers", len(feature_audit_df), "of the", df.shape[1], "safely-loaded columns.")
feature_audit_df["Decision"].value_counts()


In [ ]:
pd.set_option("display.max_rows", 50)
feature_audit_df


### Output Interpretation

Denetim, 44 güvenli kolonun **12'sinde doğrulanmış bir çift/kopya veya sabit değer** (`Decision`
sütununda "Drop" olarak işaretlenenlerin çoğu) ortaya çıkardı — bunlar **varsayım değil, gerçek
eşitlik kontrolleriyle** (`.mean()==1.0` karşılaştırmaları) doğrulandı: `Benefit per order` ≡
`Order Profit Per Order`, `Category Id` ≡ `Product Category Id`, `Order Item Cardprod Id` ≡
`Product Card Id`, `Product Price` ≡ `Order Item Product Price`, `Product Status` sabit (`0`).
Bu, "her yeni kolonun otomatik olarak yeni sinyal taşıdığı" varsayımının yanlış olabileceğinin
somut bir kanıtıdır.

### Part B — Direct Target Leakage

`Delivery Status` alanını ve `actual_shipping_days > scheduled_shipping_days` koşulunu, **sadece
training verisi üzerinde**, target ile ilişkisi açısından inceleyelim.


In [ ]:
print("Cross-tabulation: Delivery Status vs. Target (training data would show the same pattern --")
print("this relationship holds across the entire dataset, verified below):")
print(pd.crosstab(df["Delivery Status"], df[TARGET]))


**Bulgular:** `Delivery Status == "Late delivery"` olan **her** satır `Late_delivery_risk == 1`;
diğer üç kategori (`Advance shipping`, `Shipping canceled`, `Shipping on time`) ise **her zaman**
`Late_delivery_risk == 0`. Bu, bire bir (deterministik) bir ilişkidir — `Delivery Status`,
hedefin yalnızca **başka bir kodlamasıdır**, bu yüzden Part A'da "Drop" kararı alındı.

### Önemli — "0" Her Zaman Başarılı Teslimat Anlamına Gelmez


In [ ]:
shipping_canceled_count = (df["Delivery Status"] == "Shipping canceled").sum()
print(f"'Shipping canceled' rows: {shipping_canceled_count}")
print(f"Of these, risk=0: {(df.loc[df['Delivery Status'] == 'Shipping canceled', TARGET] == 0).sum()} "
      f"(100% -- verified, not assumed)")


**7,754 satır** `"Shipping canceled"` olarak işaretli ve bunların **tamamı** `risk=0` (gecikme
riski yok) etiketine sahip. **Bu yanıltıcıdır** — bir sevkiyatın iptal edilmesi, onun "başarılı ve
zamanında" tamamlandığı anlamına gelmez; veri kümesinin etiketleme kuralı basitçe bu durumu
"gecikme riski yok" olarak kodlamıştır. Bu LAB'da bu nüansı **düzeltmiyoruz** (veri kümesinin
orijinal etiketini değiştirmiyoruz), ancak bu sınırlamayı açıkça not ediyoruz.

### `actual_shipping_days > scheduled_shipping_days` Koşulu — Hedefi Tam Açıklıyor mu?


In [ ]:
shipping_delay_condition = df["Days for shipping (real)"] > df["Days for shipment (scheduled)"]
condition_vs_target = pd.crosstab(shipping_delay_condition, df[TARGET])
condition_vs_target.index = ["actual <= scheduled", "actual > scheduled"]
condition_vs_target


**Bu koşul hedefi tamamen açıklamıyor:** `actual <= scheduled` olduğunda risk **her zaman** `0`
(tutarlı); ancak `actual > scheduled` olan satırların **4,423'ü** (toplam 103,400 satırın küçük
bir kısmı) hâlâ `risk=0` olarak etiketli. Bu satırların önemli bir kısmı muhtemelen iptal edilmiş
sevkiyatlardır (yukarıdaki `Shipping canceled` grubuyla kısmen örtüşür). **Bu koşulun "hedefi tam
olarak açıkladığı" iddia edilmemelidir** — kısmen açıklayan, ama istisnaları olan bir ilişkidir.

### Optional Demonstration — `INVALID — TARGET LEAKAGE`

Aşağıdaki küçük, **izole** model, `Delivery Status`'u (bilinçli olarak, sadece bu demonstrasyon
için) bir feature olarak kullanıyor. Bu model **ana benchmark'a dahil edilmeyecek, production
adayı olarak sunulmayacak ve test verisinde değerlendirilmeyecektir** — yalnızca leakage'ın
pratikte ne kadar yanıltıcı bir performans görüntüsü yaratabileceğini göstermek içindir. `Delivery
Status` ile target arasındaki ilişki **tam olarak deterministik** olduğu için (Part B'de
gösterildi), beklenti mükemmele yakın değil, **tam olarak mükemmel** bir skordur.


In [ ]:
# INVALID — TARGET LEAKAGE. Demonstration only. Never used in the main benchmark (Section 5),
# never evaluated on the test set, never reused in later sections or later LABs.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

leakage_demo_X = df[["Delivery Status", "Order Item Quantity"]].copy()
leakage_demo_y = df[TARGET]

X_demo_train, X_demo_val, y_demo_train, y_demo_val = train_test_split(
    leakage_demo_X, leakage_demo_y, test_size=0.2, stratify=leakage_demo_y, random_state=RANDOM_STATE
)

leakage_demo_preprocessor = ColumnTransformer([
    ("delivery_status", OneHotEncoder(handle_unknown="ignore"), ["Delivery Status"]),
], remainder="passthrough")

leakage_demo_pipe = Pipeline(steps=[
    ("preprocessing", leakage_demo_preprocessor),
    ("model", XGBClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS)),
]).fit(X_demo_train, y_demo_train)

demo_proba = leakage_demo_pipe.predict_proba(X_demo_val)[:, 1]
demo_pred = (demo_proba >= 0.5).astype(int)
print("[INVALID -- TARGET LEAKAGE] Val Accuracy:", round(accuracy_score(y_demo_val, demo_pred), 4))
print("[INVALID -- TARGET LEAKAGE] Val ROC-AUC :", round(roc_auc_score(y_demo_val, demo_proba), 4))
print()
print("This perfect score is NOT a real model -- it is Delivery Status leaking the label back")
print("into the feature matrix. Compare against Section 5's real benchmark (~0.5-0.6 ROC-AUC range).")


### Part C — Group-Aware Temporal Data Splitting

Bu veri kümesinde sipariş tarihleri var ve bazı siparişler birden fazla order item içeriyor
(Section 1'de doğrulandı: ortalama ~2.75 kayıt/sipariş). Bu yüzden **rastgele satır bazlı split
varsayılan yaklaşım olmamalıdır** — aynı siparişin bazı kalemleri train'e, bazıları val/test'e
düşerse, bu kalemler **bağımsız gözlemler değildir** (aynı siparişin ortak özelliklerini
paylaşırlar) ve bu durum sızıntıya benzer bir optimistik yanlılık yaratabilir.

### Hands-On — Datetime Dönüşümü


In [ ]:
df["order_datetime"] = pd.to_datetime(df["order date (DateOrders)"], format="%m/%d/%Y %H:%M")

print("order_datetime range:", df["order_datetime"].min(), "->", df["order_datetime"].max())
print()
print("Rows per year:")
print(df["order_datetime"].dt.year.value_counts().sort_index())


2018 yılı sadece **2,123 satır** içeriyor (Ocak 2018'in küçük bir kuyruğu) — veri kümesinin
doğal bitiş noktası budur.

### Hands-On — Aynı Siparişin Tüm Kalemleri Aynı Tarihe Sahip mi?


In [ ]:
order_date_span = df.groupby("Order Id")["order_datetime"].agg(["min", "max"])
order_date_span["span_days"] = (order_date_span["max"] - order_date_span["min"]).dt.total_seconds() / 86400

print("Maximum order_datetime span within a single Order Id (days):", order_date_span["span_days"].max())


**Doğrulandı:** Aynı `Order Id`'ye ait tüm order item'lar **birebir aynı** `order_datetime`'a
sahiptir (maksimum span = 0 gün). Bu, basit bir **satır-bazlı kronolojik kesim**'in bile otomatik
olarak sipariş bütünlüğünü koruyacağı anlamına gelir — çünkü aynı siparişin tüm satırları, kesim
noktasının **aynı tarafında** kalacaktır. Bunu aşağıda **varsaymıyoruz, doğruluyoruz**.

### Hands-On — Kronolojik Kesim Noktalarının Hesaplanması

`%70/%15/%15` oranlarına karşılık gelen tarih kesim noktalarını, satır sayısına göre
**quantile tabanlı** olarak (veriden deterministik biçimde) hesaplıyoruz.


In [ ]:
df_sorted_by_date = df.sort_values("order_datetime")
n_rows = len(df_sorted_by_date)

train_ratio = config["lab08"]["split"]["train_ratio"]
validation_ratio = config["lab08"]["split"]["validation_ratio"]

train_cutoff = df_sorted_by_date["order_datetime"].iloc[int(n_rows * train_ratio)]
validation_cutoff = df_sorted_by_date["order_datetime"].iloc[int(n_rows * (train_ratio + validation_ratio))]

print("Train cutoff      :", train_cutoff)
print("Validation cutoff :", validation_cutoff)
print()
print("NOTE: these cutoffs are a DETERMINISTIC function of this dataset (quantiles of the sorted")
print("order_datetime column) -- re-running this exact cell on the same data always reproduces the")
print("same cutoffs. LAB09 can therefore recreate this split by re-running this computation, with")
print("no separate persisted split file required.")


In [ ]:
train_mask = df["order_datetime"] <= train_cutoff
validation_mask = (df["order_datetime"] > train_cutoff) & (df["order_datetime"] <= validation_cutoff)
test_mask = df["order_datetime"] > validation_cutoff

train_df = df.loc[train_mask].copy()
validation_df = df.loc[validation_mask].copy()
test_df = df.loc[test_mask].copy()

print(f"Train      : {len(train_df):7,d} rows ({len(train_df) / n_rows:.4f})")
print(f"Validation : {len(validation_df):7,d} rows ({len(validation_df) / n_rows:.4f})")
print(f"Test       : {len(test_df):7,d} rows ({len(test_df) / n_rows:.4f})")


### Hands-On — Split Bütünlük Kontrolleri


In [ ]:
train_orders = set(train_df["Order Id"])
validation_orders = set(validation_df["Order Id"])
test_orders = set(test_df["Order Id"])

print("Row overlap checks (must all be 0):")
print("  train/validation index overlap:", len(set(train_df.index) & set(validation_df.index)))
print("  train/test index overlap      :", len(set(train_df.index) & set(test_df.index)))
print("  validation/test index overlap :", len(set(validation_df.index) & set(test_df.index)))
print()
print("Order Id overlap checks (must all be 0 -- this is the group-aware guarantee):")
print("  train/validation Order Id overlap:", len(train_orders & validation_orders))
print("  train/test Order Id overlap      :", len(train_orders & test_orders))
print("  validation/test Order Id overlap :", len(validation_orders & test_orders))
print()
print("Temporal ordering check:")
print("  train max date      <= validation min date? ",
      train_df["order_datetime"].max() <= validation_df["order_datetime"].min())
print("  validation max date <= test min date?       ",
      validation_df["order_datetime"].max() <= test_df["order_datetime"].min())


In [ ]:
print("Date ranges:")
for name, part in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"  {name:10s}: {part['order_datetime'].min()} -> {part['order_datetime'].max()}")
print()
print("Class distribution (target positive rate):")
for name, part in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"  {name:10s}: {part[TARGET].mean():.4f}  (n={len(part):,d}, unique Order Id={part['Order Id'].nunique():,d})")


### Output Interpretation

Bütün bütünlük kontrolleri **gerçekten** `0` çıktı — satır çakışması yok, **Order Id çakışması
yok** (group-aware garanti sağlandı), ve kronolojik sıralama korunuyor. Üç kümede de hedef pozitif
oranı stabil (~%54-55 aralığında) — belirgin bir temporal class drift gözlemlenmiyor.

### Label Availability — Bir Sınırlama Notu

Bu veri kümesi **statik, geçmişe dönük (historical)** bir veri kümesidir — her satırın
`Late_delivery_risk` etiketi, veri toplandığı anda **zaten bilinen, sonuçlanmış** bir değerdir
(canlı/akan bir sistemde "henüz sonucu bilinmeyen sipariş" durumu burada söz konusu değildir).
Bu yüzden, kesim noktasına yakın siparişler için "etiketin henüz bilinip bilinmediği" sorusu bu
statik veri kümesi bağlamında **uygulanamaz** — bu, LAB'ın canlı bir sistemle çalıştığı
**varsayılmadığı** anlamına gelir; gerçek bir operasyonel sistemde bu varsayımın ayrıca
doğrulanması gerekir.

### Test Set Strategy

**LAB08 ve LAB09 aynı veri kümesini ve aynı split'i kullanacaktır.** Bu LAB boyunca:

- Model ve feature selection kararları **yalnızca `train_df`/`validation_df` üzerinden**
  verilecektir.
- `test_df` üzerinde **hiçbir** metrik hesaplanmayacak, tahmin üretilmeyecek, feature importance
  veya SHAP analizi yapılmayacaktır.
- `test_df`, LAB09'un sonundaki **final evaluation** için saklı tutulmaktadır.

Split cutoff'ları (`train_cutoff`, `validation_cutoff`) yukarıda açıklandığı gibi veriden
deterministik olarak türetildiği için, LAB09 bu hücreleri yeniden çalıştırarak **aynı split'i**
üretebilir — ayrı bir persisted split dosyasına ihtiyaç yoktur.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: validation_df ve test_df'in Late_delivery_risk oranlarını train_df'inkiyle karşılaştırın.
# TODO: Bu üç oranın birbirine ne kadar yakın olduğunu yorumlayın -- bu, temporal class drift'in
#       ciddi bir sorun olup olmadığına dair bir kanıttır.

# print(train_df[TARGET].mean(), validation_df[TARGET].mean(), test_df[TARGET].mean())


---


## Section 3 — Feature Engineering Techniques
**Süre: 20 dakika**

### Part A — Baseline Feature Set

Prediction-time availability açısından makul olan domain feature'larını, Section 2 Part A'daki
Feature Audit'e dayanarak seçiyoruz.


In [ ]:
baseline_features = [
    "Days for shipment (scheduled)",
    "Shipping Mode",
    "Market",
    "Order Region",
    "Customer Segment",
    "Department Name",
    "Category Name",
    "Order Item Quantity",
    "Order Item Product Price",
    "Order Item Discount Rate",
]

missing_from_df = [c for c in baseline_features if c not in df.columns]
print("All baseline features present in dataframe?", len(missing_from_df) == 0)
print("baseline_features:", baseline_features)


**Eğitim varsayımı (her feature için):** Bu 10 feature'ın tümü, siparişin verildiği anda
(sevkiyat öncesinde) bilindiği varsayılmaktadır — `Days for shipment (scheduled)` planlanan
süredir (gerçekleşen değil); `Shipping Mode` müşterinin/sistemin seçtiği yöntemdir; diğerleri
ürün/müşteri/sipariş özellikleridir. **Bu varsayımların gerçek bir operasyonel sistemde ayrıca
doğrulanması gerekir.**

### Önemli — Erken Bir Redundancy Bulgusu

Section 2'de `Days for shipment (scheduled)`'i Keep, `Shipping Mode`'u da Keep olarak işaretledik
— ama bu ikisi arasındaki ilişkiyi henüz incelemedik.


In [ ]:
print("Days for shipment (scheduled) vs Shipping Mode cross-tabulation:")
print(pd.crosstab(train_df["Shipping Mode"], train_df["Days for shipment (scheduled)"]))


**Bulgu:** `Shipping Mode` ve `Days for shipment (scheduled)` arasında **tam (1-e-1)** bir
ilişki var — `Same Day`→`0`, `First Class`→`1`, `Second Class`→`2`, `Standard Class`→`4`,
başka hiçbir kombinasyon yok. Bu, **leakage değildir** (her ikisi de sipariş anında bilinir) ama
**ciddi bir redundancy**'dir — bu ikisi, pratikte **aynı bilgiyi** iki farklı temsille taşıyor.
Section 4'teki correlation/feature selection analizinde bu bulguya geri döneceğiz. Şimdilik
ikisini de baseline'da tutuyoruz (farklı temsiller bazı modeller için farklı şekilde yararlı
olabilir) ama bu redundancy'yi not ediyoruz.

### Part B — Date-Time Feature Engineering


In [ ]:
def create_datetime_features(frame: pd.DataFrame) -> pd.DataFrame:
    # Deterministic, row-level Date-Time features derived from order_datetime.
    # Each output value depends only on that row's own timestamp -- no statistic is learned
    # from the dataset, so this function is safe to apply identically to train/validation/test.
    frame = frame.copy()
    frame["order_month"] = frame["order_datetime"].dt.month
    frame["order_day_of_week"] = frame["order_datetime"].dt.day_of_week
    frame["order_hour"] = frame["order_datetime"].dt.hour
    frame["order_quarter"] = frame["order_datetime"].dt.quarter
    frame["is_weekend"] = (frame["order_datetime"].dt.day_of_week >= 5).astype(int)
    return frame


train_df = create_datetime_features(train_df)
validation_df = create_datetime_features(validation_df)
test_df = create_datetime_features(test_df)

print("Datetime parsing failures (NaT) in order_datetime:")
for name, part in [("Train", train_df), ("Validation", validation_df), ("Test", test_df)]:
    print(f"  {name:10s}: {part['order_datetime'].isna().sum()}")


**Datetime parsing hataları sessizce kabul edilmiyor** — yukarıdaki hücre, her bölmede `NaT`
(parse edilemeyen tarih) sayısını açıkça raporluyor; `0` çıkması, bütün tarihlerin
`%m/%d/%Y %H:%M` formatıyla başarıyla ayrıştırıldığını **doğrular**.

### Hands-On — Date-Time Feature'ların Target ile İlişkisi (sadece training verisinde)


In [ ]:
print("Target rate by is_weekend:")
print(train_df.groupby("is_weekend")[TARGET].mean())
print()
print("Target rate by order_day_of_week (0=Monday):")
print(train_df.groupby("order_day_of_week")[TARGET].mean())
print()
print("Target rate by order_month:")
print(train_df.groupby("order_month")[TARGET].mean())


### Interpretation

`is_weekend` (0.5491 vs 0.5503), `order_day_of_week` (0.542-0.558 aralığı) ve `order_month`
(0.537-0.558 aralığı) için target oranları **çok dar bir aralıkta** değişiyor — bu, bu üç
feature'ın **tek başına zayıf bir sinyal** taşıdığını gösteriyor. Bu, bu feature'ların **otomatik
olarak** faydalı olacağı varsayımının yanlış olabileceğinin bir örneğidir; gerçek katkılarını
Section 5'in kontrollü benchmark'ında (E01 vs E02) ölçeceğiz.

### `order_year` Hakkında Bir Not

`order_year`'ı **kasıtlı olarak** feature set'imize dahil etmiyoruz. Training verisi 2015-2017
yıllarını kapsıyor, validation/test ise 2017-2018'e uzanıyor (Section 2 Part C). Eğer model
`order_year`'ı güçlü bir sinyal olarak öğrenirse, bu sinyal **gelecekteki (görülmemiş) yıllara
genellenemeyebilir** — model, zaman içinde kayan bir örüntüyü değil, belirli yılların özelliklerini
ezberleme riski taşır. `order_month`/`order_quarter` gibi **döngüsel** zaman feature'ları bu
riski taşımaz, çünkü her yıl tekrar eder.

### Part C — Cyclical Encoding

Saat gibi döngüsel değişkenlerde, ham sayısal değer (`order_hour ∈ {0, ..., 23}`) yanıltıcıdır:
model, `23` ve `0` saatlerinin birbirine **çok yakın** olduğunu (günün aynı bölümü) doğrudan
göremez — `23` ve `0` arasındaki ham sayısal fark (`23`) maksimumdur.

**Matematiksel Gösterim:**

$$x_{sin}=\sin\left(\frac{2\pi x}{T}\right)$$

$$x_{cos}=\cos\left(\frac{2\pi x}{T}\right)$$

Burada `T` döngünün periyodu (saat için `24`).


In [ ]:
for part_name, part in [("train_df", train_df), ("validation_df", validation_df), ("test_df", test_df)]:
    part["order_hour_sin"] = np.sin(2 * np.pi * part["order_hour"] / 24)
    part["order_hour_cos"] = np.cos(2 * np.pi * part["order_hour"] / 24)

print("order_hour=23 and order_hour=0 raw difference:", abs(23 - 0))
print("order_hour=23 and order_hour=0 cyclical (sin, cos) distance:")
h23 = (np.sin(2 * np.pi * 23 / 24), np.cos(2 * np.pi * 23 / 24))
h0 = (np.sin(2 * np.pi * 0 / 24), np.cos(2 * np.pi * 0 / 24))
print(f"  hour=23 -> (sin={h23[0]:.4f}, cos={h23[1]:.4f})")
print(f"  hour=0  -> (sin={h0[0]:.4f}, cos={h0[1]:.4f})")
print(f"  Euclidean distance in (sin, cos) space: {np.sqrt((h23[0]-h0[0])**2 + (h23[1]-h0[1])**2):.4f}")


Ham değerlerde `23` ile `0` arasındaki fark **maksimum** (`23`) iken, cyclical encoding'de bu
ikisi birbirine **çok yakın** (`(sin, cos)` uzayında küçük bir Euclidean distance) — bu, saat
`23:00` ile `00:00`'ın gerçekte birbirine yakın olduğu sezgisini doğru yansıtıyor.

Cyclical Encoding'in **özellikle linear modellerde** (mesafe/ağırlıklı toplam tabanlı) faydalı
olabileceğini unutmayın — **tree-based modellerde** (bu LAB'ın Section 5'te kullanacağı XGBoost
dahil) her zaman bir performans artışı sağlayacağı **varsayılamaz**, çünkü ağaç tabanlı modeller
zaten sıralı bir değişkeni (`order_hour`) birden fazla eşik ile bölerek dolaylı olarak
döngüselliğe yakın davranışlar öğrenebilir. Bu LAB, bu iddiayı **Section 5'in gerçek
benchmark'ında** test edecektir.

### Part D — Numerical & Interaction Features


In [ ]:
def create_interaction_features(frame: pd.DataFrame) -> pd.DataFrame:
    # Deterministic, row-level numerical/interaction features (no fitting required).
    frame = frame.copy()
    frame["gross_item_value"] = frame["Order Item Quantity"] * frame["Order Item Product Price"]
    # Order Item Discount Rate was verified in Section 2 to already be a 0-1 fraction (max 0.25),
    # so multiplying it directly by gross_item_value is scale-safe.
    frame["estimated_discount_amount"] = frame["gross_item_value"] * frame["Order Item Discount Rate"]
    frame["quantity_x_scheduled_days"] = frame["Order Item Quantity"] * frame["Days for shipment (scheduled)"]
    frame["log_gross_item_value"] = np.log1p(frame["gross_item_value"])
    return frame


train_df = create_interaction_features(train_df)
validation_df = create_interaction_features(validation_df)
test_df = create_interaction_features(test_df)

print("gross_item_value <= 0 rows (would break log1p):", (train_df["gross_item_value"] <= 0).sum())
train_df[["gross_item_value", "estimated_discount_amount", "quantity_x_scheduled_days", "log_gross_item_value"]].describe()


### Hands-On — Bu Yeni Feature'lar Gerçekten "Yeni" Sinyal mi Taşıyor?

`estimated_discount_amount`'ı, veri kümesindeki (baseline/engineered set'imizde kullanmadığımız)
ham `Order Item Discount` sütunuyla karşılaştıralım.


In [ ]:
discount_correlation = train_df["estimated_discount_amount"].corr(train_df["Order Item Discount"])
print(f"Correlation(estimated_discount_amount, Order Item Discount) = {discount_correlation:.6f}")


**Bulgu:** Korelasyon **0.9999** — yani `estimated_discount_amount`, veri kümesinde zaten var
olan (ama bizim baseline/engineered set'imize dahil etmediğimiz) `Order Item Discount` sütununu
**pratik olarak yeniden keşfediyor**. Bu, mühendislik ürünü bir feature'ın her zaman "tamamen
yeni" bir sinyal taşımadığının; bazen ham veride zaten var olan bilgiyi farklı bir yoldan yeniden
türettiğinin somut bir örneğidir.

### Hands-On — Interaction Candidate: Weekend × Shipping Mode

`is_weekend` ile `Shipping Mode`'un **birlikte** target üzerinde, her ikisinin ayrı ayrı
etkisinden farklı bir örüntü oluşturup oluşturmadığını inceleyelim.


In [ ]:
print("Target rate by is_weekend x Shipping Mode:")
print(train_df.groupby(["is_weekend", "Shipping Mode"])[TARGET].mean().unstack())


**Bulgu:** Her `Shipping Mode` içinde, `is_weekend=0` ve `is_weekend=1` arasındaki target oranı
**neredeyse özdeş** (örn. `Standard Class`: %38.1 vs %38.1; `First Class`: %95.7 vs %95.4) — yani
hafta sonu bilgisi, `Shipping Mode`'un etkisini **anlamlı biçimde değiştirmiyor**. **Bu interaction
candidate'ı kasıtlı olarak final engineered feature set'imize dahil etmiyoruz** — bu, "her
interaction candidate'ının otomatik olarak faydalı olduğu" varsayımının gerçek verilerle
çürütüldüğü bir örnektir.

Bunun yerine, en güçlü ham sinyali not edelim: **`Shipping Mode`, tek başına, target oranında
`%38.1` (`Standard Class`) ile `%95.3` (`First Class`) arasında değişen çarpıcı bir fark
yaratıyor** — bu, bu dataset'teki en güçlü aday feature'dır (Section 6'da feature importance ile
tekrar karşılaşacağız).

### Part E — Feature Quality Validation


In [ ]:
engineered_feature_columns = [
    "order_month", "order_day_of_week", "order_quarter", "is_weekend",
    "order_hour_sin", "order_hour_cos",
    "gross_item_value", "estimated_discount_amount", "quantity_x_scheduled_days", "log_gross_item_value",
]

quality_report_rows = []
for col in engineered_feature_columns:
    series = train_df[col]
    quality_report_rows.append({
        "Feature": col,
        "Missing": int(series.isna().sum()),
        "Infinite": int(np.isinf(series).sum()) if pd.api.types.is_numeric_dtype(series) else 0,
        "Constant": bool(series.nunique() <= 1),
        "Nunique": int(series.nunique()),
    })

quality_report_df = pd.DataFrame(quality_report_rows)
quality_report_df


**Doğrulandı:** Hiçbir engineered feature'da missing value, infinite value veya constant
(sıfır varyans) değer yok. `engineered_feature_columns` listesi, `order_hour`'un ham hâlini
**değil**, yalnızca `order_hour_sin`/`order_hour_cos`'u içeriyor (ham `order_hour`'u cyclical
encoding'den sonra ayrıca kullanmıyoruz, çünkü bu bilgiyi zaten taşıyor).

### Engineering Rule — Deterministic vs. Data-Dependent Transformations

Bu section'daki **bütün** transformasyonlar (tarih parçalama, cyclical encoding, çarpım/log) her
satır için **bağımsız ve deterministiktir** — bir satırın dönüşümü, diğer satırlara **hiç
bakmadan** hesaplanabilir. Bu yüzden `train_df`/`validation_df`/`test_df`'e **aynı fonksiyonu**
ayrı ayrı uygulamak güvenlidir (`create_datetime_features`, `create_interaction_features`).

**Bunun karşıtı — data-dependent transformation:** Örneğin bir "mean target encoding" (her
`Category Name` için, o kategorideki ortalama `Late_delivery_risk` oranını bir feature olarak
kullanmak) **tüm veri kümesinden öğrenilen bir istatistiktir** — bu tür bir dönüşüm **yalnızca
training verisinden** fit edilmeli, validation/test'e sadece `transform` edilmelidir (LAB04'teki
`fit`/`transform` ayrımıyla aynı prensip). Bu LAB'da böyle bir data-dependent encoding
**kullanmıyoruz** — ama bu farkı açıkça not ediyoruz, çünkü Section 4'teki Feature Selection
yöntemleri (örn. `SelectKBest`) tam olarak bu kategoriye girecek.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: order_quarter'a göre target oranını hesaplayın (train_df.groupby("order_quarter")[TARGET].mean()).
# TODO: Bu oranın order_month'a göre hesapladığımız orana kıyasla daha mı az yoksa daha mı çok
#       değişkenlik gösterdiğini yorumlayın.

# print(train_df.groupby("order_quarter")[TARGET].mean())


---


## Section 4 — Feature Selection Methods
**Süre: 20 dakika**

### Part A — Feature Selection Taxonomy

### Learning Objective

Filter, Wrapper ve Embedded Method'ların çalışma prensiplerini, avantajlarını ve sınırlamalarını
kavramsal olarak anlamak.

### Concept Explanation

- **Filter Methods**: Feature'ları, **model bağımsız** istatistiksel bir ölçütle (varyans,
  korelasyon, ANOVA F-test, Mutual Information) değerlendirir. Hızlıdır, ama feature'lar
  arasındaki etkileşimleri göz ardı eder.
- **Wrapper Methods**: Feature alt kümelerini, **gerçek bir modeli tekrar tekrar eğiterek**
  değerlendirir (örn. Recursive Feature Elimination). Model-feature etkileşimini yakalar, ama
  **hesaplama maliyeti çok yüksektir**.
- **Embedded Methods**: Feature selection, **modelin eğitim sürecine gömülüdür** (örn. L1
  regularization'ın katsayıları sıfırlaması, ya da bir ağaç tabanlı modelin
  `feature_importances_`'ı). Wrapper'dan daha hızlı, Filter'dan daha model-duyarlıdır.

| Yöntem | Avantaj | Sınırlama |
|---|---|---|
| Filter (Variance, Correlation, F-test, MI) | Hızlı, model bağımsız | Feature interaction'ları ölçemez |
| Wrapper (RFE, Forward/Backward) | Model-feature etkileşimini yakalar | Çok yüksek hesaplama maliyeti |
| Embedded (L1, Tree-based, SelectFromModel) | Hız/model-duyarlılık dengesi | Seçim, kullanılan modele özgüdür |

Ana LAB'da **düşük hesaplama maliyetli** yöntemleri (Filter + bir Embedded demo) tercih ediyoruz;
Wrapper Method'lar (RFE) hesaplama maliyeti nedeniyle **Advanced Challenge 1**'e bırakılıyor.

### Part B — Variance Threshold


In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Demonstration on ALL original safe numeric columns (not yet filtered) -- this lets the TOOL
# itself (re)discover what Section 2's manual audit already found by inspection.
all_numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

variance_selector = VarianceThreshold(threshold=config["lab08"]["feature_selection"]["variance_threshold"])
variance_selector.fit(train_df[all_numeric_columns].fillna(0))

zero_variance_columns = [
    col for col, keep in zip(all_numeric_columns, variance_selector.get_support()) if not keep
]
print("Columns with zero variance (threshold=0.0), found by VarianceThreshold:")
print(zero_variance_columns)


**Bulgu:** `VarianceThreshold`, `Product Description` (tamamen eksik, `fillna(0)` sonrası sabit
`0`) ve `Product Status` (zaten sabit `0`) sütunlarını **bağımsız olarak** buluyor — bu, Section
2'nin manuel denetiminde zaten "Drop" kararı aldığımız iki sütunla **birebir örtüşüyor**. Düşük
varyansın **mutlaka** düşük predictive value anlamına gelmediğini unutmayın — örneğin nadir ama
güçlü bir sinyal taşıyan ikili bir feature de düşük varyansa sahip olabilir; `VarianceThreshold`
yalnızca **sabit/sıfır-varyans** durumunu güvenle eler.

### Part C — Correlation Analysis

Section 3'teki numerik feature'lar (baseline + engineered) arasındaki korelasyonu inceleyelim.


In [ ]:
engineered_extra_features = [
    "order_month", "order_day_of_week", "order_quarter", "is_weekend",
    "order_hour_sin", "order_hour_cos",
    "gross_item_value", "estimated_discount_amount", "quantity_x_scheduled_days", "log_gross_item_value",
]
numerical_features = [
    "Days for shipment (scheduled)", "Order Item Quantity", "Order Item Product Price", "Order Item Discount Rate",
] + engineered_extra_features
categorical_features = ["Shipping Mode", "Market", "Order Region", "Customer Segment", "Department Name", "Category Name"]
engineered_features = numerical_features + categorical_features

correlation_matrix = train_df[numerical_features].corr()
correlation_matrix.round(2)


In [ ]:
import matplotlib.pyplot as plt

# Color palette: fixed categorical/semantic roles reused across the ml-labs notebook series.
BAR_COLOR = "#2a78d6"
HIGHLIGHT_COLOR = "#1baf7a"
WARNING_COLOR = "#eb6834"
GRID_COLOR = "#e1e0d9"


def style_axes(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.8)
    ax.set_axisbelow(True)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(correlation_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(numerical_features)))
ax.set_yticks(range(len(numerical_features)))
ax.set_xticklabels(numerical_features, rotation=90, fontsize=7)
ax.set_yticklabels(numerical_features, fontsize=7)
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Feature Correlation Heatmap (numerical, training data)")
plt.tight_layout()
plt.show()


In [ ]:
high_correlation_pairs = []
for i, col_a in enumerate(numerical_features):
    for col_b in numerical_features[i + 1:]:
        r = correlation_matrix.loc[col_a, col_b]
        if abs(r) > 0.6:
            high_correlation_pairs.append((col_a, col_b, round(float(r), 4)))

pd.DataFrame(high_correlation_pairs, columns=["Feature_A", "Feature_B", "Correlation"])


### Interpretation

Yüksek korelasyonlu çiftlerin **neredeyse tamamı mantıklı bir kaynak-türev ilişkisini**
yansıtıyor: `order_month`↔`order_quarter` (`r=0.97`, quarter ay'dan türetiliyor),
`order_day_of_week`↔`is_weekend` (`r=0.79`, weekend day_of_week'ten türetiliyor),
`gross_item_value`↔`log_gross_item_value` (`r=0.94`, log aynı büyüklüğün monotonik dönüşümü),
`Order Item Quantity`↔`quantity_x_scheduled_days` ve `Order Item Discount Rate`↔
`estimated_discount_amount` de benzer şekilde kaynak-türev ilişkileri.

**Bu korelasyonlara dayanarak hiçbir feature'ı silmiyoruz.** Yüksek korelasyon; **redundancy**'nin
bir göstergesi olabilir, ama **causality** (nedensellik) değildir — ve ağaç tabanlı modeller
(Section 5'teki XGBoost dahil), lineer modellere kıyasla korelasyonlu feature'lara karşı
**genellikle** daha dayanıklıdır (her split'te yalnızca bir feature kullanılır). Section 5'in
E02 vs E03 karşılaştırması, bu redundant feature'ların gerçekten çıkarılmasının performansı
değiştirip değiştirmediğini **ölçecektir** — şimdiden silmiyoruz.

### Part D — SelectKBest


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif

# NOTE: no StandardScaler here -- XGBoost (Section 5's model) is split-threshold-based and does
# not require feature scaling; adding one would be an unnecessary preprocessing step.
numerical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numerical_pipeline, numerical_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    verbose_feature_names_out=False,
)

X_train_engineered = train_df[engineered_features]
y_train = train_df[TARGET]

X_train_transformed = preprocessor.fit_transform(X_train_engineered, y_train)
transformed_feature_names = preprocessor.get_feature_names_out()
print("Transformed feature count (baseline + engineered, after one-hot encoding):", X_train_transformed.shape[1])


In [ ]:
select_k_best_k = config["lab08"]["feature_selection"]["select_k_best"]
k_best_selector = SelectKBest(score_func=f_classif, k=select_k_best_k)
k_best_selector.fit(X_train_transformed, y_train)

f_test_scores = pd.Series(k_best_selector.scores_, index=transformed_feature_names).sort_values(ascending=False)
print(f"Top 10 ANOVA F-test scores (out of {len(transformed_feature_names)} transformed features):")
print(f_test_scores.head(10))


### Interpretation

`Shipping Mode_Standard Class` ve `Days for shipment (scheduled)` F-skorları (~25,552 ve ~19,754)
listenin tepesinde ve diğer feature'lardan **büyük bir farkla** ayrılıyor — bu, Section 3'te
gözlemlediğimiz "Shipping Mode en güçlü sinyal" bulgusunu **nicel olarak** doğruluyor.

**ANOVA F-test, univariate bir scoring yaklaşımıdır** — her feature'ı **tek başına**, diğer
feature'larla etkileşimini görmeden değerlendirir. Bu nedenle feature interaction'larını **tek
başına ölçemez**. F-skorlarını kesin bir istatistiksel önem veya nedensellik göstergesi olarak
**yorumlamıyoruz** — yalnızca, bu feature setindeki **göreli sıralama** için bir sinyal olarak
kullanıyoruz.


In [ ]:
selected_feature_mask = k_best_selector.get_support()
selected_feature_names = transformed_feature_names[selected_feature_mask]
print(f"SelectKBest(k={select_k_best_k}) selected {selected_feature_mask.sum()} features:")
print(list(selected_feature_names))


### Part E — Mutual Information

Mutual Information, ANOVA F-test'in **doğrusal olmayan** istatistiksel bağımlılıkları da
yakalayabilen bir alternatifidir. Hesaplama maliyeti nedeniyle, training verisinin **reproducible
bir alt örneklemi** üzerinde göstereceğiz.


In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi_sample_size = 5000
mi_sample = train_df.sample(n=mi_sample_size, random_state=RANDOM_STATE)
X_mi_sample_transformed = preprocessor.transform(mi_sample[engineered_features])
y_mi_sample = mi_sample[TARGET]

# discrete_features: True for one-hot encoded categorical columns (always 0/1) and for the
# naturally discrete numerical columns; False for genuinely continuous numerical columns.
discretely_valued = {"order_month", "order_day_of_week", "order_quarter", "is_weekend", "Order Item Quantity"}
discrete_feature_mask = [
    (name in discretely_valued) or (i >= len(numerical_features))
    for i, name in enumerate(transformed_feature_names)
]

mutual_info_scores = mutual_info_classif(
    X_mi_sample_transformed, y_mi_sample,
    discrete_features=discrete_feature_mask, random_state=RANDOM_STATE,
)
mi_series = pd.Series(mutual_info_scores, index=transformed_feature_names).sort_values(ascending=False)
print(f"Top 10 Mutual Information scores (on a {mi_sample_size}-row training sample):")
print(mi_series.head(10))


### Interpretation

Mutual Information'ın en yüksek skorlu feature'ları (`Days for shipment (scheduled)`,
`quantity_x_scheduled_days`, `Shipping Mode` dummy'leri) **ANOVA F-test ile büyük ölçüde
örtüşüyor** — iki farklı istatistiksel yaklaşımın aynı dominant sinyale işaret etmesi, bu
bulgunun **tesadüfi olmadığının** bir göstergesidir. Mutual Information hesaplamasında
continuous/discrete feature ayrımına dikkat ettik (`discrete_features` maskesi) — bu ayrımı
yanlış yapmak (örn. one-hot kolonlarını continuous olarak işaretlemek), skorları anlamlı
biçimde bozabilir.

### Part F — Embedded Feature Selection

`SelectFromModel`, bir modelin öğrendiği feature importance'ları (ağaç tabanlı modellerde
`feature_importances_`, L1-regularized modellerde katsayılar) kullanarak feature seçimi yapar.


In [ ]:
from sklearn.feature_selection import SelectFromModel
from xgboost import XGBClassifier

embedded_estimator = XGBClassifier(n_estimators=50, max_depth=3, random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS)
embedded_estimator.fit(X_train_transformed, y_train)

embedded_selector = SelectFromModel(embedded_estimator, prefit=True, threshold="median")
embedded_mask = embedded_selector.get_support()
print(f"SelectFromModel (XGBoost, threshold='median') selected {embedded_mask.sum()} / {len(embedded_mask)} features.")


Bu LAB'da kapsamlı bir embedded feature selection çalışması **zorunlu değildir** — bu kısa
demonstrasyon, `SelectFromModel`'in nasıl çalıştığını göstermek içindir. `L1-Regularized
LogisticRegression` ile benzer bir yaklaşım, deneyimli katılımcılar için **Advanced Challenge
2**'de ele alınacaktır.

### Wrapper Methods (Kavramsal)

**Recursive Feature Elimination (RFE)**, bir modeli tekrar tekrar eğitip her turda en az önemli
feature'ı çıkararak çalışır; **RFECV** buna cross-validation ekler. Her iki yöntem de, her turda
**tam bir model eğitimi** gerektirdiği için hesaplama maliyeti **çok yüksektir** — bu yüzden ana
LAB'da uygulanmıyor, **Advanced Challenge 1**'de, training verisinin kontrollü bir örnekleminde
ele alınacaktır.

### Critical Rules

- Feature Selection **yalnızca training verisi üzerinde** fit edilir (`k_best_selector.fit(...)`,
  `embedded_estimator.fit(...)` — hepsi `X_train_transformed`/`y_train` üzerinde).
- Validation/test target değerleri feature selector fitting'de **hiç kullanılmadı**.
- `SelectKBest`/`SelectFromModel`, Section 5'te bir Scikit-learn `Pipeline` içerisine
  yerleştirilecek (preprocessing → feature selection → estimator).
- Transformed feature isimleri (`get_feature_names_out()`) her aşamada korunuyor.
- One-Hot Encoding sonrası türetilen dummy'ler (örn. `Shipping Mode_First Class`) ile orijinal
  kategorik feature (`Shipping Mode`) arasındaki ilişki açıkça takip ediliyor.
- Feature sayısı, Feature Selection öncesi (**79**, yukarıda ölçüldü) ve sonrası (**20**,
  `SelectKBest` ile) açıkça raporlandı.

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: f_test_scores (ANOVA) ve mi_series (Mutual Information) sıralamalarının ilk 10 feature'ını
#       karşılaştırın.
# TODO: İki yöntemin ne kadar örtüştüğünü, hangi feature'larda ayrıştığını yorumlayın.

# print(set(f_test_scores.head(10).index) & set(mi_series.head(10).index))


---

## Break
**Süre: 10 dakika**

Kısa bir ara verin. Devam ederken Section 5'te, bu LAB'ın ana deneysel bölümünde, baseline/
engineered/selected feature set'lerini **aynı XGBoost konfigürasyonuyla** karşılaştırılacağız.

---


## Section 5 — Comparative Feature Benchmarking
**Süre: 25 dakika**

**Bu LAB'ın ana deneysel bölümüdür.**

### Objective

Model algoritmasını (XGBoost) ve hyperparametrelerini **sabit tutarak**, feature set
değişikliklerinin (baseline → engineered → selected) performans üzerindeki etkisini ölçmek.

### Primary Model


In [ ]:
from xgboost import XGBClassifier

model_cfg = config["lab08"]["model"]
print("XGBoost configuration (fixed across all three experiments, NOT claimed optimal):", model_cfg)


def make_model():
    return XGBClassifier(
        objective="binary:logistic",
        n_estimators=model_cfg["n_estimators"],
        max_depth=model_cfg["max_depth"],
        learning_rate=model_cfg["learning_rate"],
        tree_method=model_cfg["tree_method"],
        device=model_cfg["device"],
        eval_metric=model_cfg["eval_metric"],
        random_state=RANDOM_STATE,
        n_jobs=BENCHMARK_THREADS,
    )


In [ ]:
from time import perf_counter
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
)

y_val = validation_df[TARGET]
DECISION_THRESHOLD = config["lab08"]["benchmark"]["decision_threshold"]
print("y_train shape:", y_train.shape, " y_val shape:", y_val.shape)
print("Decision threshold (common to all experiments):", DECISION_THRESHOLD)


### Baseline Model


In [ ]:
from sklearn.dummy import DummyClassifier

dummy_model = DummyClassifier(strategy="most_frequent").fit(
    train_df[["Order Item Quantity"]], y_train
)
dummy_pred_val = dummy_model.predict(validation_df[["Order Item Quantity"]])
print("DummyClassifier Validation Accuracy:", round(accuracy_score(y_val, dummy_pred_val), 4))


### Experiments E01-E03

| Experiment | Feature Set | Preprocessing | Feature Selection |
|---|---|---|---|
| **E01** | `baseline_features` (10) | ColumnTransformer | None |
| **E02** | `baseline_features` + `engineered_extra_features` (20) | ColumnTransformer | None |
| **E03** | Same as E02 | ColumnTransformer | `VarianceThreshold` + `SelectKBest(k=20)` |

Her üç deney de **aynı** `make_model()` konfigürasyonunu, **aynı** training/validation split'ini,
**aynı** `random_state`'i ve **aynı** `0.5` decision threshold'unu kullanır. Model konfigürasyonu
sonuçlar görülmeden önce belirlendi — bu, kapsamlı bir hyperparameter optimizasyonu değildir.

### Experiment Integrity Note

Feature Engineering (Section 3) hiçbir satırı kaybetmedi — `train_df`/`validation_df`'in satır
sayısı Section 2'deki split'ten bu yana değişmedi (`create_datetime_features`/
`create_interaction_features` satır bazlı, deterministik dönüşümlerdir). Bu yüzden E01/E02/E03,
**tam olarak aynı gözlemler** üzerinde karşılaştırılıyor.


In [ ]:
def build_pipeline(experiment: str):
    if experiment == "E01":
        preprocessing = ColumnTransformer(
            transformers=[
                ("numerical", Pipeline([("imputer", SimpleImputer(strategy="median"))]), [
                    "Days for shipment (scheduled)", "Order Item Quantity",
                    "Order Item Product Price", "Order Item Discount Rate",
                ]),
                ("categorical", Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]), categorical_features),
            ],
            verbose_feature_names_out=False,
        )
        steps = [("preprocessing", preprocessing), ("model", make_model())]
        feature_columns = baseline_features
    elif experiment in ("E02", "E03"):
        preprocessing = ColumnTransformer(
            transformers=[
                ("numerical", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numerical_features),
                ("categorical", Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]), categorical_features),
            ],
            verbose_feature_names_out=False,
        )
        steps = [("preprocessing", preprocessing)]
        if experiment == "E03":
            steps += [
                ("variance_threshold", VarianceThreshold(threshold=config["lab08"]["feature_selection"]["variance_threshold"])),
                ("select_k_best", SelectKBest(score_func=f_classif, k=config["lab08"]["feature_selection"]["select_k_best"])),
            ]
        steps.append(("model", make_model()))
        feature_columns = engineered_features
    else:
        raise ValueError(f"Unknown experiment: {experiment}")
    return Pipeline(steps=steps), feature_columns


print("build_pipeline() ready for E01/E02/E03.")


### Timing Measurement

LAB07'deki benchmark metodolojisini yeniden kullanıyoruz: `time.perf_counter()`, training için
**3 tekrarın median'ı** (her tekrarda yeni/unfit pipeline), inference için **warm-up + 10
tekrarın median'ı**.

**Önemli — ölçüm kapsamı:** `Training_Time_s` ve `Batch_Inference_Latency_ms`, yalnızca
**preprocessing (ColumnTransformer) + feature selection (varsa) + estimator** aşamalarını
kapsar. **Date-Time ve interaction feature'ları (Section 3) bu ölçümün dışındadır** — onlar
`train_df`/`validation_df` üzerinde **bir kez, önceden** hesaplandı (Pipeline'ın parçası değil).
Bu, E02/E03'ün inference latency'sinin, bu feature'ların **sıfırdan** hesaplanma maliyetini
**içermediği** anlamına gelir. Bu sınırlamayı aşağıda ayrıca ölçüyoruz.


In [ ]:
def measure_training_time(experiment, repeats=None):
    repeats = repeats or config["lab08"]["benchmark"]["training_repeats"]
    times = []
    fitted_pipeline = None
    feature_columns = None
    for _ in range(repeats):
        pipe, feature_columns = build_pipeline(experiment)
        start = perf_counter()
        pipe.fit(train_df[feature_columns], y_train)
        times.append(perf_counter() - start)
        fitted_pipeline = pipe
    return float(np.median(times)), fitted_pipeline, feature_columns


def measure_inference_latency(fitted_pipeline, X, repeats=None, warmup=None):
    repeats = repeats or config["lab08"]["benchmark"]["inference_repeats"]
    warmup = warmup if warmup is not None else config["lab08"]["benchmark"]["warmup_runs"]
    for _ in range(warmup):
        fitted_pipeline.predict_proba(X)
    times = []
    for _ in range(repeats):
        start = perf_counter()
        fitted_pipeline.predict_proba(X)
        times.append(perf_counter() - start)
    return float(np.median(times))


### Hands-On — E01/E02/E03 Çalıştırma


In [ ]:
benchmark_rows = []
fitted_pipelines = {}

for experiment in ["E01", "E02", "E03"]:
    training_time_s, fitted_pipeline, feature_columns = measure_training_time(experiment)
    fitted_pipelines[experiment] = (fitted_pipeline, feature_columns)

    X_val_experiment = validation_df[feature_columns]
    pred_train = fitted_pipeline.predict(train_df[feature_columns])
    proba_val = fitted_pipeline.predict_proba(X_val_experiment)[:, 1]
    pred_val = (proba_val >= DECISION_THRESHOLD).astype(int)

    batch_latency_s = measure_inference_latency(fitted_pipeline, X_val_experiment)

    transformed_count = fitted_pipeline.named_steps["preprocessing"].transform(X_val_experiment.iloc[:5]).shape[1]
    if "select_k_best" in fitted_pipeline.named_steps:
        selected_count = int(fitted_pipeline.named_steps["select_k_best"].get_support().sum())
    else:
        selected_count = transformed_count

    row = {
        "Experiment": experiment,
        "Feature_Strategy": {"E01": "Baseline", "E02": "Engineered", "E03": "Selected"}[experiment],
        "Input_Feature_Count": len(feature_columns),
        "Transformed_Feature_Count": int(transformed_count),
        "Selected_Feature_Count": selected_count,
        "Train_Accuracy": accuracy_score(y_train, pred_train),
        "Validation_Accuracy": accuracy_score(y_val, pred_val),
        "Validation_Precision": precision_score(y_val, pred_val),
        "Validation_Recall": recall_score(y_val, pred_val),
        "Validation_F1": f1_score(y_val, pred_val),
        "Validation_ROC_AUC": roc_auc_score(y_val, proba_val),
        "Validation_AP": average_precision_score(y_val, proba_val),
        "Training_Time_s": training_time_s,
        "Batch_Inference_Latency_ms": batch_latency_s * 1000,
    }
    row["Generalization_Gap"] = row["Train_Accuracy"] - row["Validation_Accuracy"]
    benchmark_rows.append(row)
    print("Benchmarked:", experiment)

feature_benchmark_df = pd.DataFrame(benchmark_rows)
feature_benchmark_df.round(4)


### Feature Engineering Timing — Ayrı Ölçüm


In [ ]:
feature_engineering_repeats = 10
start = perf_counter()
for _ in range(feature_engineering_repeats):
    fe_sample = validation_df[["order date (DateOrders)", "Order Item Quantity", "Order Item Product Price", "Order Item Discount Rate", "Days for shipment (scheduled)"]].copy()
    fe_sample["order_datetime"] = pd.to_datetime(fe_sample["order date (DateOrders)"], format="%m/%d/%Y %H:%M")
    fe_sample = create_datetime_features(fe_sample)
    fe_sample["order_hour_sin"] = np.sin(2 * np.pi * fe_sample["order_hour"] / 24)
    fe_sample["order_hour_cos"] = np.cos(2 * np.pi * fe_sample["order_hour"] / 24)
    fe_sample = create_interaction_features(fe_sample)
feature_engineering_time_s = (perf_counter() - start) / feature_engineering_repeats

print(f"Feature engineering time for the full validation batch ({len(validation_df):,d} rows): "
      f"{feature_engineering_time_s * 1000:.2f} ms")
print(f"Per-sample feature engineering time: {feature_engineering_time_s * 1000 / len(validation_df):.5f} ms")
print(f"For comparison, E02's reported Batch_Inference_Latency_ms: "
      f"{feature_benchmark_df.loc[feature_benchmark_df['Experiment'] == 'E02', 'Batch_Inference_Latency_ms'].iloc[0]:.2f} ms")


### Output Interpretation

Yukarıdaki hücrenin çıktısı, feature engineering süresinin E02'nin raporlanan batch inference
latency'sine kıyasla **ne kadarlık bir ek maliyet** temsil ettiğini gösteriyor. Bu oran, Section
6/7'deki Training Time/Inference Latency ölçümleri gibi **çalıştırmalar arasında değişebilir**
(timing ölçümü gürültülüdür — Accuracy/F1/ROC-AUC gibi deterministik değildir). Önemli olan kesin
yüzde değil, **niteliksel sonuç**: **gerçek bir production sisteminde**, E02/E03'ün toplam
inference maliyeti, tablodaki `Batch_Inference_Latency_ms` değerinin **üzerinde** olacaktır (bu
ek feature engineering adımı eklenmeden) — bu LAB'ın raporladığı sayılar bu ek maliyeti
içermez; bu açıkça bir sınırlamadır, göz ardı edilmemelidir.

### Benchmark DataFrame


In [ ]:
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
feature_benchmark_df


### Interpretation

- **Feature Engineering performansı artırdı mı?** Evet — E02 (Validation ROC-AUC ≈ 0.749), E01'e
  (≈ 0.720) kıyasla **gerçek bir iyileşme** gösteriyor (+0.03 ROC-AUC puanı), Validation F1'de de
  benzer bir iyileşme var (0.649 → 0.683).
- **Feature Selection feature sayısını azalttı mı?** Evet, **belirgin biçimde** — E03, E02'nin 79
  transformed feature'ından sadece **20'sini** (`Selected_Feature_Count`) kullanıyor (**%75
  azalma**).
- **Daha az feature ile benzer performans elde edilebilir mi?** Evet — E03 (20 feature),
  Validation ROC-AUC'de (≈ 0.751) E02'den (79 feature, ≈ 0.749) **daha düşük değil, hatta hafifçe
  daha yüksek**. Bu, bu dataset için feature sayısının dramatik biçimde azaltılmasının performans
  kaybına yol açmadığını gösteriyor.
- **Ek feature'lar inference maliyetini artırdı mı?** E01→E02'de evet (daha fazla transformed
  feature, biraz daha yüksek latency); E02→E03'te ise **daha az** final feature'a rağmen latency
  **hafifçe arttı** (ek `VarianceThreshold`/`SelectKBest` transform adımlarının overhead'i) — bu,
  "daha az feature = her zaman daha hızlı inference" varsayımının **her zaman doğru olmadığının**
  bir örneğidir.
- **Hangi feature strategy daha iyi bir engineering trade-off sağlıyor?** Bu çalıştırmada **E03**
  (Selected) — E02 ile neredeyse aynı (hatta hafifçe daha iyi) predictive performansı, **4 kat
  daha az** feature ile sağlıyor. Bu, daha basit, daha bakımı kolay bir model demektir — ancak
  training/inference time'daki küçük farklar, bu üçünün arasındaki seçimi **kesin olarak**
  belirlemiyor; Section 7'nin iş senaryolarında bu farklı önceliklere göre değerlendirilecek.

### Required Visualizations


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(feature_benchmark_df["Experiment"], feature_benchmark_df["Validation_ROC_AUC"], color=BAR_COLOR)
ax.axhline(0.5, color=WARNING_COLOR, linewidth=1.5, linestyle="--", label="Random guessing (0.5)")
ax.set_title("Validation ROC-AUC by Experiment")
ax.set_ylabel("Validation ROC-AUC")
ax.legend(frameon=False)
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(feature_benchmark_df["Experiment"], feature_benchmark_df["Validation_F1"], color=HIGHLIGHT_COLOR)
ax.set_title("Validation F1 by Experiment")
ax.set_ylabel("Validation F1")
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
x_pos = np.arange(len(feature_benchmark_df))
width = 0.3
ax.bar(x_pos - width, feature_benchmark_df["Input_Feature_Count"], width, label="Input", color=BAR_COLOR)
ax.bar(x_pos, feature_benchmark_df["Transformed_Feature_Count"], width, label="Transformed", color=HIGHLIGHT_COLOR)
ax.bar(x_pos + width, feature_benchmark_df["Selected_Feature_Count"], width, label="Selected (final)", color=WARNING_COLOR)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_benchmark_df["Experiment"])
ax.set_title("Feature Count by Experiment")
ax.set_ylabel("Feature Count")
ax.legend(frameon=False)
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(feature_benchmark_df["Experiment"], feature_benchmark_df["Training_Time_s"], color=BAR_COLOR)
ax.set_title("Training Time by Experiment (median of 3 runs)")
ax.set_ylabel("Training Time (s)")
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(feature_benchmark_df["Experiment"], feature_benchmark_df["Batch_Inference_Latency_ms"], color=HIGHLIGHT_COLOR)
ax.set_title("Inference Latency by Experiment (median of 10 runs, full validation batch)")
ax.set_ylabel("Batch Inference Latency (ms)")
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for _, row in feature_benchmark_df.iterrows():
    ax.scatter(row["Selected_Feature_Count"], row["Validation_ROC_AUC"], s=100, color=BAR_COLOR, zorder=3)
    ax.annotate(row["Experiment"], (row["Selected_Feature_Count"], row["Validation_ROC_AUC"]),
                textcoords="offset points", xytext=(8, 4))
ax.set_xlabel("Final Feature Count")
ax.set_ylabel("Validation ROC-AUC")
ax.set_title("Performance vs Feature Count")
style_axes(ax)
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
x_pos = np.arange(len(feature_benchmark_df))
width = 0.35
ax.bar(x_pos - width / 2, feature_benchmark_df["Train_Accuracy"], width, label="Train Accuracy", color=BAR_COLOR)
ax.bar(x_pos + width / 2, feature_benchmark_df["Validation_Accuracy"], width, label="Validation Accuracy", color=WARNING_COLOR)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_benchmark_df["Experiment"])
ax.set_title("Training vs Validation Accuracy")
ax.set_ylabel("Accuracy")
ax.legend(frameon=False)
style_axes(ax)
plt.tight_layout()
plt.show()


### Student Exercise


In [ ]:
# Student Exercise:
# TODO: feature_benchmark_df'teki Generalization_Gap sütununu inceleyin.
# TODO: E01'den E03'e doğru gap'in nasıl değiştiğini yorumlayın -- daha fazla feature mutlaka
#       daha fazla overfitting'e mi yol açıyor?

# print(feature_benchmark_df[["Experiment", "Generalization_Gap"]])


---


## Section 6 — Feature Importance & SHAP Explainability
**Süre: 25 dakika**

Bu section, LAB08'in **ikinci temel uygulama alanıdır.**

### Model Seçimi — Hangi Pipeline Açıklanacak?

Section 5'in üç deneyinden **E03 (Selected)**'i açıklıyoruz: E02 ile neredeyse aynı (hatta
hafifçe daha iyi) performansı, **4 kat daha az feature** ile sağladığı için hem hesaplama
açısından daha hafif (SHAP için önemli) hem de yorumlaması daha basit. Bu açık bir seçim
kararıdır — varsayılmamıştır.

### Part A — Model-Based Feature Importance

### Concept Explanation

XGBoost üç farklı importance türü sunar:

- **Weight**: Bir feature'ın, ormandaki ağaçlarda **kaç kez split noktası olarak kullanıldığı**.
- **Gain**: Bir feature'ın kullanıldığı split'lerin **ortalama kayıp azalımına katkısı**.
- **Cover**: Bir feature'ın kullanıldığı split'lerin etkilediği **ortalama örnek sayısı
  (Hessian ağırlıklı)**.

### Important

Model-Based Feature Importance:

- Feature'ın **nedensel** etkisini göstermez.
- Feature'ın prediction üzerindeki etki **yönünü** doğrudan göstermez (sadece "ne kadar
  kullanıldığını" gösterir, "pozitif mi negatif mi" göstermez).
- **Correlated Features** nedeniyle yanıltıcı olabilir (iki korelasyonlu feature, önemi
  paylaşarak her biri ayrı ayrı daha "önemsiz" görünebilir).
- **Modelden modele** (hatta importance türünden türe) değişebilir.


In [ ]:
fitted_pipeline_e03, feature_columns_e03 = fitted_pipelines["E03"]
fitted_model = fitted_pipeline_e03.named_steps["model"]

# Extract the final, selected feature names (after preprocessing -> variance_threshold -> select_k_best)
all_transformed_names = fitted_pipeline_e03.named_steps["preprocessing"].get_feature_names_out()
after_variance_names = all_transformed_names[fitted_pipeline_e03.named_steps["variance_threshold"].get_support()]
selected_feature_names = after_variance_names[fitted_pipeline_e03.named_steps["select_k_best"].get_support()]

fitted_model.get_booster().feature_names = list(selected_feature_names)
print("Model sees", len(selected_feature_names), "features:", list(selected_feature_names))


In [ ]:
def extract_selected_feature_names(pipeline):
    # Returns the final feature names a fitted E03-style pipeline's model actually sees.
    names = pipeline.named_steps["preprocessing"].get_feature_names_out()
    names = names[pipeline.named_steps["variance_threshold"].get_support()]
    names = names[pipeline.named_steps["select_k_best"].get_support()]
    return names


importance_types = ["weight", "gain", "cover"]
booster = fitted_model.get_booster()
top_by_type = {}
for importance_type in importance_types:
    scores = booster.get_score(importance_type=importance_type)
    top_by_type[importance_type] = pd.Series(scores).sort_values(ascending=False)
    print(f"Top 5 by '{importance_type}':")
    print(top_by_type[importance_type].head(5))
    print()


### Interpretation

Üç importance türü **farklı** "en önemli feature"ı gösteriyor: `weight`'e göre `order_hour_sin`
(en çok kullanılan split), `gain`'e göre `Days for shipment (scheduled)` (en çok kayıp azaltan),
`cover`'a göre `Shipping Mode_First Class` (en çok örneği etkileyen). **Bu, "feature importance
modelden modele (ve importance türünden türe) değişebilir" uyarısının somut bir kanıtıdır.**
Aşağıda, Scikit-learn'ün varsayılan `feature_importances_`'ını (gain-tabanlı, normalize edilmiş)
görselleştiriyoruz.


In [ ]:
feature_importance_series = pd.Series(
    fitted_model.feature_importances_, index=selected_feature_names
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
top_n = feature_importance_series.head(10).sort_values()
ax.barh(top_n.index, top_n.values, color=BAR_COLOR)
ax.set_title("Model-Based Feature Importance (XGBoost, gain-normalized), Top 10")
ax.set_xlabel("Importance (feature_importances_)")
style_axes(ax)
plt.tight_layout()
plt.show()


### Part B — Permutation Importance

Permutation Importance, bir feature'ı **rastgele karıştırıp** (shuffle) model performansının
ne kadar düştüğünü ölçer — model-agnostic bir yöntemdir (XGBoost'a özgü değildir).

**Matematiksel Gösterim:**

$$I_j=s-\frac{1}{K}\sum_{k=1}^{K}s_{k,j}$$

- $s$: Orijinal model skoru
- $s_{k,j}$: Feature $j$ permute edildikten sonraki model skoru ($k$. tekrar)
- $K$: Tekrar sayısı

**Önemli tasarım kararı:** Permutation'ı, preprocessing'den **önceki orijinal feature
kolonları** üzerinde, fitted pipeline'ın tamamı üzerinden uyguluyoruz (`Shipping Mode` gibi bir
kategorik feature'ı tek bir birim olarak karıştırıyoruz) — bunun yerine one-hot encoded dummy
kolonlarını **ayrı ayrı** permute etseydik, `Shipping Mode_First Class=1` ve
`Shipping Mode_Same Day=1` gibi **gerçekte var olmayan** kombinasyonlar (bağımlı dummy'lerin
birbirinden bağımsız karıştırılması) oluşabilirdi.


In [ ]:
from sklearn.inspection import permutation_importance

permutation_sample_size = config["lab08"]["explainability"]["permutation_sample_size"]
permutation_repeats = config["lab08"]["explainability"]["permutation_repeats"]

permutation_sample = validation_df.sample(n=permutation_sample_size, random_state=RANDOM_STATE)
X_permutation_sample = permutation_sample[feature_columns_e03]
y_permutation_sample = permutation_sample[TARGET]

permutation_result = permutation_importance(
    fitted_pipeline_e03, X_permutation_sample, y_permutation_sample,
    scoring="roc_auc", n_repeats=permutation_repeats, random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS,
)

permutation_importance_df = pd.DataFrame({
    "Feature": feature_columns_e03,
    "Importance_Mean": permutation_result.importances_mean,
    "Importance_Std": permutation_result.importances_std,
}).sort_values("Importance_Mean", ascending=False)

permutation_importance_df


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
top_perm = permutation_importance_df.head(10).sort_values("Importance_Mean")
ax.barh(top_perm["Feature"], top_perm["Importance_Mean"], xerr=top_perm["Importance_Std"], color=HIGHLIGHT_COLOR)
ax.set_title("Permutation Importance (ROC-AUC drop), Top 10")
ax.set_xlabel("Importance Mean (+/- Std over 5 repeats)")
style_axes(ax)
plt.tight_layout()
plt.show()


### Interpretation

En önemli üç feature (`Days for shipment (scheduled)`, `order_hour_sin`, `Shipping Mode`) Part
A'daki bulgularla **genel olarak örtüşüyor** — farklı bir yöntem (model-agnostic, raw kolonlar
üzerinde) aynı dominant sinyale işaret ediyor.

**Negatif importance değeri:** `Order Region` için importance mean'i **hafifçe negatif**
(`≈-0.0004`, standart sapması da küçük `≈0.0002`) — yani bu feature'ı karıştırmak, bazen modeli
**çok hafif iyileştiriyor** gibi görünüyor. Bu, gerçek bir "zararlı feature" anlamına gelmez;
`Order Region`'un gerçek sinyali sıfıra çok yakın olduğunda, 5 tekrarlık ölçüm gürültüsü
pozitif/negatif yönde rastgele sapabilir. **Bu değeri otomatik olarak silmiyoruz** — bu, zayıf
sinyalli feature'larda beklenen bir istatistiksel gürültü örüntüsüdür.

**Correlated Features uyarısı:** `Days for shipment (scheduled)` ile `Shipping Mode` (Section 3'te
tam 1-e-1 ilişkili olduğu gösterilmişti) birbirine **çok yakın** importance değerlerine sahip —
bu iki feature birbirinin sinyalini "paylaşıyor" olabilir; bireysel importance değerleri, bu
redundancy nedeniyle gerçek etkilerinden **düşük görünebilir**.

### Part C — SHAP Fundamentals

**SHAP (SHapley Additive exPlanations)**, **Shapley Values** (Cooperative Game Theory'den gelen
bir kavram) kullanarak, bir feature'ın **tek bir tahmine** yaptığı katkıyı adil biçimde paylaştırır
— her feature, "oyuna" katılan bir "oyuncu" gibi ele alınır ve farklı feature kombinasyonları
altındaki marjinal katkısı ortalanır.

**Basitleştirilmiş Additive Explanation:**

$$f(x)=\phi_0+\sum_{i=1}^{M}\phi_i$$

- $f(x)$: Açıklanan model çıktısı (bu LAB'da **raw margin**, olasılık değil — bkz. Critical SHAP
  Rules)
- $\phi_0$: Base value (tüm açıklama örnekleminin ortalama çıktısı)
- $\phi_i$: Feature $i$'nin katkısı
- $M$: Feature sayısı

Feature katkıları, **açıklanan model output space'i içerisinde** yorumlanmalıdır — aşağıda bu
space'in ne olduğunu açıkça belirtiyoruz.

### Part D — SHAP Implementation


In [ ]:
import shap

X_train_model_input = fitted_pipeline_e03.named_steps["select_k_best"].transform(
    fitted_pipeline_e03.named_steps["variance_threshold"].transform(
        fitted_pipeline_e03.named_steps["preprocessing"].transform(train_df[feature_columns_e03])
    )
)
X_val_model_input = fitted_pipeline_e03.named_steps["select_k_best"].transform(
    fitted_pipeline_e03.named_steps["variance_threshold"].transform(
        fitted_pipeline_e03.named_steps["preprocessing"].transform(validation_df[feature_columns_e03])
    )
)

print("No transformer or selector was re-fit here -- all three .transform() calls reuse the")
print("fitted_pipeline_e03 state learned in Section 5 (from training data only).")
print("X_train_model_input shape:", X_train_model_input.shape, " X_val_model_input shape:", X_val_model_input.shape)


### SHAP Sampling Strategy

Bütün 180,519 kayıt üzerinde SHAP hesaplamıyoruz — bu, CPU tabanlı eğitim ortamında gereksiz
hesaplama maliyeti yaratır. `config/settings.yaml`'daki `lab08.explainability` bloğundan
yapılandırılabilir örneklem boyutları kullanıyoruz.


In [ ]:
explainability_cfg = config["lab08"]["explainability"]
rng = np.random.default_rng(RANDOM_STATE)

background_indices = rng.choice(X_train_model_input.shape[0], size=explainability_cfg["shap_background_size"], replace=False)
background_sample = X_train_model_input[background_indices]

explanation_indices = rng.choice(X_val_model_input.shape[0], size=explainability_cfg["shap_sample_size"], replace=False)
explanation_sample = X_val_model_input[explanation_indices]
explanation_sample_df = pd.DataFrame(explanation_sample, columns=selected_feature_names)

print("Background sample size (training):", background_sample.shape[0])
print("Explanation sample size (validation):", explanation_sample.shape[0])
print("shap version:", shap.__version__)

explainer = shap.TreeExplainer(fitted_model, model_output="raw")
shap_values = explainer(explanation_sample_df)
print("shap_values.values shape:", shap_values.values.shape)


### Critical SHAP Rules

XGBoost binary classification için `model_output="raw"` kullandık — bu, açıklanan değerin
**log-odds (raw margin)** uzayında olduğu anlamına gelir, **olasılık değil**. Bir feature için
SHAP değerinin `+0.5` olması, "gecikme olasılığının 50 yüzde puan arttığı" anlamına **gelmez** —
bu değer, sigmoid'den **önceki** log-odds skalasındadır.


In [ ]:
from scipy.special import expit

raw_margin_predictions = fitted_model.predict(explanation_sample, output_margin=True)
reconstructed_from_shap = shap_values.base_values + shap_values.values.sum(axis=1)

max_reconstruction_error = np.max(np.abs(reconstructed_from_shap - raw_margin_predictions))
print(f"Additivity check -- max |base_value + sum(SHAP) - raw_margin_prediction|: {max_reconstruction_error:.2e}")
print("(Should be ~0, up to floating point precision -- confirms SHAP values are additive and")
print(" consistent with the model's actual raw output for these exact rows.)")


In [ ]:
probability_from_margin = expit(raw_margin_predictions)
probability_direct = fitted_model.predict_proba(explanation_sample)[:, 1]

print("First 3 rows -- raw margin vs. sigmoid(raw margin) vs. predict_proba():")
for i in range(3):
    print(f"  row {i}: raw_margin={raw_margin_predictions[i]:.4f}  "
          f"sigmoid(raw_margin)={probability_from_margin[i]:.4f}  "
          f"predict_proba={probability_direct[i]:.4f}")


**Doğrulandı:** `sigmoid(raw margin) == predict_proba()` (floating point hassasiyeti dahilinde) —
bu, raw margin ve probability arasındaki ilişkiyi somut biçimde gösteriyor. SHAP contributions'ı
yorumlarken **her zaman** hangi output space'te olduğumuzu (`raw` margin) belirtiyoruz.

### Part E — Global Explainability


In [ ]:
shap.plots.bar(shap_values)


**Mean Absolute SHAP Value** (bar grafiğindeki değerler), her feature'ın **ortalama mutlak
katkısını** gösterir — bir feature'ın pozitif/negatif yönünü değil, genel büyüklüğünü özetler.


In [ ]:
shap.plots.beeswarm(shap_values)


### Interpretation

Beeswarm grafiği, her gözlem için feature katkısının **yönünü ve büyüklüğünü** (renk = feature
değeri, x ekseni = SHAP değeri) gösteriyor — bar grafiğinin aksine, **tek bir sayıya**
indirmiyor. Bu, **Global** (bar: tüm örneklemin ortalama önem sıralaması) ile **Local** (her
noktanın kendi katkısı) açıklama arasındaki farkı somutlaştırıyor. One-hot encoded kategorik
feature'ları (örn. `Shipping Mode_First Class`) yorumlarken, bunun ikili (0/1) bir değer
olduğunu ve "yüksek/düşük" renk kodlamasının bu ikili durumu yansıttığını unutmayın.

### Part F — Local Explainability

Validation örnekleminden üç gözlem seçiyoruz: **yüksek tahmin edilen risk**, **düşük tahmin
edilen risk**, ve **threshold'a yakın, yanlış sınıflandırılmış** bir gözlem.


In [ ]:
predicted_probabilities = fitted_model.predict_proba(explanation_sample)[:, 1]
predicted_labels = (predicted_probabilities >= DECISION_THRESHOLD).astype(int)
true_labels = y_val.values[explanation_indices]

high_risk_position = int(np.argmax(predicted_probabilities))
low_risk_position = int(np.argmin(predicted_probabilities))

misclassified_positions = np.where(predicted_labels != true_labels)[0]
print(f"Misclassified observations in this 500-row explanation sample: {len(misclassified_positions)}")

if len(misclassified_positions) > 0:
    distance_to_threshold = np.abs(predicted_probabilities - DECISION_THRESHOLD)
    near_threshold_position = int(misclassified_positions[np.argmin(distance_to_threshold[misclassified_positions])])
else:
    # Fallback (not expected here): closest-to-threshold observation, correctly labeled as such.
    near_threshold_position = int(np.argmin(np.abs(predicted_probabilities - DECISION_THRESHOLD)))

local_examples = {
    "HIGH RISK": high_risk_position,
    "LOW RISK": low_risk_position,
    "MISCLASSIFIED (near threshold)": near_threshold_position,
}

for label, position in local_examples.items():
    print(f"{label:32s} | true={true_labels[position]}  pred={predicted_labels[position]}  "
          f"predicted_probability={predicted_probabilities[position]:.4f}")


In [ ]:
for label, position in local_examples.items():
    print(f"=== {label} (position {position}) ===")
    print(f"Actual Label         : {true_labels[position]}")
    print(f"Predicted Label      : {predicted_labels[position]}")
    print(f"Predicted Probability: {predicted_probabilities[position]:.4f}")
    print(f"Base Value (raw margin space): {shap_values.base_values[position]:.4f}")
    row_shap = shap_values.values[position]
    positive_total = row_shap[row_shap > 0].sum()
    negative_total = row_shap[row_shap < 0].sum()
    print(f"Sum of Positive SHAP Contributions: {positive_total:.4f}")
    print(f"Sum of Negative SHAP Contributions: {negative_total:.4f}")
    print(f"Final Model Output (raw margin)    : {shap_values.base_values[position] + row_shap.sum():.4f}")
    print()


In [ ]:
shap.plots.waterfall(shap_values[local_examples["HIGH RISK"]])


In [ ]:
shap.plots.waterfall(shap_values[local_examples["LOW RISK"]])


In [ ]:
shap.plots.waterfall(shap_values[local_examples["MISCLASSIFIED (near threshold)"]])


### Interpretation

Üç waterfall grafiği de, base value'dan başlayarak her feature'ın (pozitif = riski artıran,
negatif = riski azaltan) katkısıyla nihai model çıktısına **nasıl ulaşıldığını** gösteriyor.
**Misclassified** örnekte, modelin güveni (predicted probability, `0.5`'e yakın) zaten düşük —
bu, modelin bu gözlem için **belirsiz** olduğunu, "kesin ama yanlış" bir tahmin yapmadığını
gösteriyor; bu tür sınır-yakın hatalar, uzak ve kesin hatalardan daha **az şaşırtıcıdır**.

### Part G — Explainability Method Comparison


In [ ]:
gain_ranking = pd.Series(fitted_model.feature_importances_, index=selected_feature_names).sort_values(ascending=False)
permutation_ranking = permutation_importance_df.set_index("Feature")["Importance_Mean"].sort_values(ascending=False)
shap_ranking = pd.Series(np.abs(shap_values.values).mean(axis=0), index=selected_feature_names).sort_values(ascending=False)

print("Top 5 -- XGBoost Gain Importance (transformed/one-hot feature granularity):")
print(gain_ranking.head(5))
print()
print("Top 5 -- Permutation Importance (ORIGINAL feature granularity):")
print(permutation_ranking.head(5))
print()
print("Top 5 -- Mean |SHAP value| (transformed/one-hot feature granularity):")
print(shap_ranking.head(5))


### Interpretation

Bu üç yöntem **farklı sorulara** cevap veriyor:

- **Gain Importance**: Feature, modelin ağaç split'lerine ne kadar katkı sağlıyor?
- **Permutation Importance**: Feature bozulduğunda (shuffle) model performansı ne kadar değişiyor?
- **Mean |SHAP|**: Feature, bireysel model tahminlerine **ortalama olarak** ne büyüklükte katkı
  sağlıyor?

**Granularity uyarısı:** `Gain`/`SHAP` sıralamaları **one-hot encoded (transformed)** feature
seviyesinde (örn. `Shipping Mode_First Class` ayrı bir satır), `Permutation` sıralaması ise
**orijinal (raw)** feature seviyesinde (`Shipping Mode` tek bir satır). Bu iki granularity'yi
**doğrudan aynı tabloda karşılaştırmıyoruz** — yukarıda ayrı ayrı sunduk. Üçü de **aynı üç ana
sinyale** (`Days for shipment (scheduled)`, `Shipping Mode`, `order_hour_sin`/
`quantity_x_scheduled_days`) işaret ediyor, ama **birebir aynı sıralamayı üretmek zorunda
değiller** — ve üretmiyorlar da.

### Explainability Limitations

- **Explainability nedensellik değildir.** SHAP, modelin **ne öğrendiğini** açıklar; gerçek
  dünyadaki gecikmenin **gerçek nedenini** belirlemez.
- **SHAP, fairness (adillik) garanti etmez.** Bir feature'ın SHAP katkısı düşük olsa da, model
  dolaylı yollarla ayrımcı davranabilir.
- **SHAP, model doğruluğunu garanti etmez.** Yanlış/zayıf bir model de tutarlı SHAP açıklamaları
  üretebilir — açıklamanın var olması, modelin doğru olduğu anlamına gelmez.
- **Feature importance model-bağımlıdır** — Part A'da gösterdiğimiz gibi, farklı importance
  türleri (hatta farklı modeller) farklı sıralamalar üretebilir.
- **Correlated features attribution'ı karmaşıklaştırır** — Part B'de gösterildi.
- **Explainability, data leakage'ı otomatik olarak ortadan kaldırmaz.** Section 2'deki
  `Delivery Status` leakage'ı, SHAP ile "açıklanabilir" olurdu (çok yüksek SHAP değeri gösterirdi)
  — ama bu, onu geçerli bir feature yapmazdı.
- **Local açıklamalar, explainer konfigürasyonuna ve referans dağılımına bağlıdır** — farklı bir
  background sample veya farklı bir explainer algoritması, aynı gözlem için farklı SHAP değerleri
  üretebilir.

Modelin bir tahmine nasıl ulaştığını açıklamak ile, gerçek dünyadaki gecikmenin gerçek nedenini
belirlemek **aynı problem değildir.**

### Student Exercise


In [ ]:
# Student Exercise:
# TODO: shap_ranking (Mean |SHAP|) ve gain_ranking (XGBoost Gain) sıralamalarının ilk 5
#       feature'ını karşılaştırın.
# TODO: Bu iki yöntemin ne kadar örtüştüğünü yorumlayın.

# print(set(shap_ranking.head(5).index) & set(gain_ranking.head(5).index))


---


## Section 7 — Final Workshop & Engineering Evaluation
**Süre: 15 dakika**

### Workshop Objective

Bu LAB boyunca gerçekleştirdiğiniz feature deneylerini ve explainability sonuçlarını
değerlendirerek teknik kararlar alacaksınız.

### Student Tasks

**1-5: Benchmark Sonuçları (Section 5'ten)**


In [ ]:
print("1) Baseline (E01) performansı:")
print(feature_benchmark_df.loc[feature_benchmark_df["Experiment"] == "E01",
      ["Validation_Accuracy", "Validation_F1", "Validation_ROC_AUC"]])

print()
print("2) Engineered (E02) vs Baseline (E01) karşılaştırması:")
e01_auc = feature_benchmark_df.loc[feature_benchmark_df["Experiment"] == "E01", "Validation_ROC_AUC"].iloc[0]
e02_auc = feature_benchmark_df.loc[feature_benchmark_df["Experiment"] == "E02", "Validation_ROC_AUC"].iloc[0]
print(f"   Val ROC-AUC: E01={e01_auc:.4f} -> E02={e02_auc:.4f}  (delta={e02_auc - e01_auc:+.4f})")

print()
print("3) Feature Selection (E03) sonrası feature sayısı:")
e02_count = feature_benchmark_df.loc[feature_benchmark_df["Experiment"] == "E02", "Transformed_Feature_Count"].iloc[0]
e03_count = feature_benchmark_df.loc[feature_benchmark_df["Experiment"] == "E03", "Selected_Feature_Count"].iloc[0]
print(f"   E02 transformed feature count: {e02_count}  ->  E03 selected feature count: {e03_count}  "
      f"({(1 - e03_count / e02_count) * 100:.1f}% reduction)")

print()
print("4) ROC-AUC ve F1 değişimi (E01 -> E02 -> E03):")
print(feature_benchmark_df[["Experiment", "Validation_ROC_AUC", "Validation_F1"]])

print()
print("5) Training/Inference maliyeti karşılaştırması:")
print(feature_benchmark_df[["Experiment", "Training_Time_s", "Batch_Inference_Latency_ms"]])


**6: En Önemli 5 Feature**


In [ ]:
print("6) En önemli 5 feature (Permutation Importance, orijinal feature granülaritesinde):")
print(permutation_importance_df.head(5)[["Feature", "Importance_Mean"]])


**7-9: Leakage, SHAP Beeswarm ve Waterfall Geri Değerlendirmesi**

**7) Potansiyel leakage riskleri:** Section 2'de `Delivery Status`'un target ile **deterministik**
olarak eşleştiğini (ve bu yüzden Drop edildiğini), `Order Status`'un `CANCELED`/
`SUSPECTED_FRAUD` kategorilerinde `risk=0` ile %100 eşleştiğini (bu yüzden "Review" — varsayılan
feature set'ine dahil edilmediğini) doğrulamıştık. Section 5-6'nın hiçbir deneyinde bu
feature'lar kullanılmadı — bu LAB'ın sonuçları bu riskten **arınmıştır**.

**8) SHAP Beeswarm yorumu:** Beeswarm grafiği, `Days for shipment (scheduled)` ve `Shipping
Mode` dummy'lerinin **hem büyüklük hem yön açısından** en etkili feature'lar olduğunu gösterdi
— bu, Section 5/6'nın bütün diğer bulgularıyla (F-test, Mutual Information, Gain, Permutation)
**tutarlıdır**.

**9) SHAP Waterfall yorumu:** Misclassified (threshold'a yakın) örnekte, modelin tahmini
(`≈0.39`) `0.5` eşiğine **yakındı** — bu, modelin "kesin ama yanlış" değil, "belirsiz" bir hata
yaptığını gösteriyor. Bu tür sınır-yakın hatalar, modelin genel güvenilirliği açısından uzak/kesin
hatalardan daha az kritiktir.

### Scenario A — High Predictive Performance


In [ ]:
scenario_a = feature_benchmark_df.loc[feature_benchmark_df["Validation_ROC_AUC"].idxmax()]
print(f"Seçilen: {scenario_a['Experiment']} ({scenario_a['Feature_Strategy']})  "
      f"Val ROC-AUC={scenario_a['Validation_ROC_AUC']:.4f}")


### Scenario B — Operational Efficiency

**Düşük feature count VE düşük inference latency önceliklidir.** Bu iki kriter bu deneyde
**çelişiyor** (Section 5'te not edildi): E03 en az final feature'a (`20`) sahip ama en **yüksek**
latency'ye; E01 en az latency'ye sahip ama en çok final feature'a (`69`). Her iki görüşü de
gösterelim.


In [ ]:
scenario_b_by_latency = feature_benchmark_df.loc[feature_benchmark_df["Batch_Inference_Latency_ms"].idxmin()]
scenario_b_by_feature_count = feature_benchmark_df.loc[feature_benchmark_df["Selected_Feature_Count"].idxmin()]

print(f"'Efficiency' = en düşük latency  -> {scenario_b_by_latency['Experiment']} "
      f"(latency={scenario_b_by_latency['Batch_Inference_Latency_ms']:.2f} ms)")
print(f"'Efficiency' = en az final feature -> {scenario_b_by_feature_count['Experiment']} "
      f"(selected_feature_count={scenario_b_by_feature_count['Selected_Feature_Count']})")


**Bu, "operasyonel verimlilik" tanımının tek bir metriğe indirgenemeyeceğinin somut bir
örneğidir** — gerçek bir mühendislik kararı, hangi kaynağın (hesaplama süresi mi, model
karmaşıklığı/bakım yükü mü) daha kritik olduğuna bağlıdır.

### Scenario C — Explainability & Maintainability

**Feature'ların anlaşılabilirliği, güvenilirliği ve bakım kolaylığı önceliklidir.** Burada da bir
nüans var: **E03** en az sayıda **modelin gördüğü** feature'a sahip (20) — bu, SHAP/Permutation
analizini daha basit hale getirir (Section 6'da zaten E03'ü bu nedenle seçtik). Ancak **E01**, en
az sayıda **ham (raw) input** feature'ına sahip (10) — bu da upstream veri pipeline'ının bakımını
basitleştirir (daha az kaynak kolonun doğru/güncel tutulması gerekir). Her iki açıdan da geçerli
bir "maintainability" argümanı var; bu LAB, **E03**'ü (zaten Section 6'da kullandığımız model)
tercih ediyor, çünkü **model-facing** karmaşıklık (açıklanması gereken feature sayısı), bu
senaryonun önceliğine daha doğrudan karşılık geliyor.

### Interpretation

Üç senaryo **farklı** feature strategy'leri haklı çıkarabiliyor (Scenario A/C → E03; Scenario B →
E01 **veya** E03, kritere bağlı). **Bu, LAB07'nin model seçimi workshop'unda gördüğümüz aynı
prensibin feature strategy seçimine uygulanmış hâlidir** — "en iyi" diye tek bir evrensel cevap
yoktur.

### Evaluation Integrity

Bu LAB'daki **bütün** kararlar (feature engineering, feature selection, model seçimi,
explainability) **yalnızca training ve validation verisi** üzerinden verildi. `test_df`
**hiçbir** metrik hesaplamasında, feature importance'ta veya SHAP analizinde kullanılmadı —
LAB09'un sonundaki final evaluation için **saklı tutulmaktadır**.


In [ ]:
print("Confirming test_df was never touched in this notebook beyond its creation in Section 2:")
print("test_df shape:", test_df.shape)
print("test_df columns include engineered features (created in Section 3, never scored):",
      all(c in test_df.columns for c in engineered_extra_features))


---

## Feature Engineering & Explainability Report

### Business Problem

**Pre-Dispatch Late Delivery Risk Prediction** — DataCo SMART Supply Chain veri kümesinde, bir
siparişin sevkiyatından önce gecikme riski taşıyıp taşımadığını tahmin etmek.

### Dataset Description

DataCo SMART Supply Chain (Kaggle: `shashwatwork/dataco-smart-supply-chain-for-big-data-analysis`,
orijinal kaynak Mendeley Data, DOI `10.17632/8gx2fvg2k6.5`). 180,519 order item satırı, 53 ham
kolon (44'ü gizlilik-güvenli biçimde yüklendi — 9 PII kolonu dosya okuma aşamasında hariç
tutuldu). Target: `Late_delivery_risk` (orta derecede dengesiz, ~%54.8/%45.2).

### Prediction Time

Sipariş bilgisi ve planlanan sevkiyat yöntemi bilinir durumdadır, ancak gerçek sevkiyat/teslimat
sonuçları henüz bilinmemektedir — bu, gerçek bir operasyonel sistemde **ayrıca doğrulanması
gereken** bir eğitim varsayımıdır.

### Data Quality Findings

44 güvenli kolonun denetiminde: 1 kolon %100 eksik (`Product Description`), 1 kolon sabit
(`Product Status`), 4 çift **tam kopya** kolon (`Benefit per order`≡`Order Profit Per Order`,
`Category Id`≡`Product Category Id`, `Order Item Cardprod Id`≡`Product Card Id`, `Product
Price`≡`Order Item Product Price`) — hepsi gerçek eşitlik kontrolleriyle doğrulandı.

### Privacy Considerations

9 PII kolonu (`Customer Email`, `Customer Password`, `Customer Fname`, `Customer Lname`,
`Customer Street`, `Customer Zipcode`, `Order Zipcode`, `Customer Id`, `Order Customer Id`)
`src/data_loading.py` seviyesinde, dosya okuma aşamasında hariç tutuldu — hiçbir zaman belleğe
yüklenmedi. Ek olarak `Latitude`/`Longitude`, kesin konum riski nedeniyle Drop edildi.

### Leakage Findings

`Delivery Status`, target ile **deterministik olarak** eşleşiyor (doğrulandı, Drop edildi).
`shipping date (DateOrders)` ve `Days for shipping (real)`, tanım olarak sevkiyat sonrası
bilgilerdir (Drop edildi). `Order Status`'un bazı kategorileri (`CANCELED`, `SUSPECTED_FRAUD`)
target ile %100 eşleşiyor ama prediction-time availability'si belirsiz — varsayılan feature
set'ine dahil edilmedi (Review). İzole bir `INVALID — TARGET LEAKAGE` demonstrasyonu, bu
leakage'ın pratikte tam (`1.0`) bir sahte performans ürettiğini gösterdi.

### Split Strategy

Group-aware kronolojik split (`%70/%15/%15`), `Order Id` bütünlüğü korunarak — doğrulandı: 0
satır/Order Id çakışması, stabil hedef oranı (~%54-55). `test_df`, LAB09 için saklı tutuluyor.

### Baseline Features

10 feature: `Days for shipment (scheduled)`, `Shipping Mode`, `Market`, `Order Region`,
`Customer Segment`, `Department Name`, `Category Name`, `Order Item Quantity`, `Order Item
Product Price`, `Order Item Discount Rate`.

### Engineered Features

Baseline + 10 ek feature: `order_month`, `order_day_of_week`, `order_quarter`, `is_weekend`,
`order_hour_sin`, `order_hour_cos`, `gross_item_value`, `estimated_discount_amount`,
`quantity_x_scheduled_days`, `log_gross_item_value`.

### Selected Features

`SelectKBest(k=20)` ile seçilen 20 transformed feature — `Days for shipment (scheduled)`,
`Shipping Mode` dummy'leri ve `quantity_x_scheduled_days` baskın.

### Experiment Results


In [ ]:
feature_benchmark_df[[
    "Experiment", "Feature_Strategy", "Input_Feature_Count", "Transformed_Feature_Count",
    "Selected_Feature_Count", "Validation_Accuracy", "Validation_F1", "Validation_ROC_AUC",
    "Training_Time_s", "Batch_Inference_Latency_ms",
]].round(4)


### Global Explainability

`Days for shipment (scheduled)` ve `Shipping Mode` dummy'leri, Gain Importance, Permutation
Importance ve Mean |SHAP| yöntemlerinin **üçünde de** en etkili feature'lar arasında — bu
tutarlılık, bulgunun istatistiksel bir tesadüf olmadığını gösteriyor.

### Local Explainability

Üç örnek (yüksek risk, düşük risk, threshold'a yakın yanlış sınıflandırma) incelendi; her üçünde
de `base_value + Σ(SHAP) = raw margin` eşitliği doğrulandı (floating point hassasiyeti dahilinde).

### Selected Feature Strategy

Bu LAB, **E03 (Selected)**'i varsayılan aday olarak öneriyor — E02 ile istatistiksel olarak
ayırt edilemeyen bir performans (Val ROC-AUC 0.7510 vs 0.7492), **4 kat daha az** final feature
ile.

### Selection Rationale

Seçim yalnızca **validation verisi** üzerinden, önceden tanımlanmış bir ranking metric (ROC-AUC)
ve tamamlayıcı metriklerle (F1, feature count, training time, inference latency) yapıldı.
Scenario B'nin gösterdiği gibi, "verimlilik" tanımı metriğe bağlı olarak E01'i de haklı
çıkarabilir — bu, nihai kararın iş önceliğine bağlı olduğunu doğrular.

### Limitations

- Hyperparametreler kapsamlı olarak optimize edilmedi (LAB09'un konusu).
- Test set bu LAB'da hiç kullanılmadı — raporlanan bütün sonuçlar development/validation
  benchmark'ıdır.
- `Benefit per order`/`Order Item Profit Ratio` gibi finansal alanların prediction-time
  availability'si doğrulanamadı (Review durumunda kaldı, kullanılmadı).
- Training Time/Inference Latency ölçümleri bu spesifik donanım için geçerlidir; evrensel
  performans sonucu değildir.

### Recommendations for LAB09

- E03 (Selected) feature strategy'sini başlangıç noktası olarak kullanın.
- Aynı `train_df`/`validation_df`/`test_df` split'ini (Section 2 Part C'deki deterministik
  cutoff'larla) yeniden oluşturun.
- Kapsamlı hyperparameter optimizasyonu (GridSearchCV/RandomizedSearchCV/Optuna) burada
  uygulanabilir.
- `test_df` üzerinde **tek seferlik** bir final evaluation planlayın.


---


## Advanced Challenges (İsteğe Bağlı)

Aşağıdaki beş challenge, ana **150 dakikalık** programın **dışındadır**. Deneyimli katılımcılar
için tasarlanmıştır. **Bu challenge'larda test set kullanılmaz.**

### Challenge 1 — Recursive Feature Elimination

RFE'yi, hesaplama maliyetini kontrol etmek için training verisinin **kontrollü bir örneklemi**
üzerinde uygulayalım.


In [ ]:
from sklearn.feature_selection import RFE

rfe_sample_size = 8000
rfe_sample = train_df.sample(n=rfe_sample_size, random_state=RANDOM_STATE)
X_rfe_sample = preprocessor.transform(rfe_sample[engineered_features])
y_rfe_sample = rfe_sample[TARGET]

start = perf_counter()
rfe_estimator = XGBClassifier(
    n_estimators=30, max_depth=model_cfg["max_depth"], tree_method=model_cfg["tree_method"],
    device=model_cfg["device"], random_state=RANDOM_STATE, n_jobs=BENCHMARK_THREADS,
)
rfe_selector = RFE(estimator=rfe_estimator, n_features_to_select=20, step=5)
rfe_selector.fit(X_rfe_sample, y_rfe_sample)
rfe_time_s = perf_counter() - start

rfe_selected_names = set(transformed_feature_names[rfe_selector.support_])
print(f"RFE time ({rfe_sample_size}-row sample, 30 trees, step=5): {rfe_time_s:.2f}s")
print("RFE selected features:", rfe_selected_names)


In [ ]:
select_k_best_names_full = set(transformed_feature_names[k_best_selector.get_support()])
rfe_vs_skb_overlap = rfe_selected_names & select_k_best_names_full
print(f"Overlap between RFE ({rfe_sample_size}-row sample) and SelectKBest (full training data, "
      f"Section 4): {len(rfe_vs_skb_overlap)} / 20")
print("Overlapping features:", rfe_vs_skb_overlap)


### Interpretation

RFE (8,000 satırlık bir örneklemde) ve `SelectKBest` (tüm training verisinde), 20 feature'lık
seçimlerinin yalnızca **5'inde** örtüşüyor. Bu, feature selection sonuçlarının **kullanılan
yönteme ve veri örneklemine duyarlı** olduğunun somut bir kanıtıdır — "doğru" bir tek feature
alt kümesi yoktur; farklı yöntemler, farklı (ama genellikle dominant sinyali — `Days for
shipment (scheduled)`, `Shipping Mode` dummy'leri, `order_hour_sin` — paylaşan) alt kümeler
üretebilir.

### Challenge 2 — Embedded Feature Selection (L1 Regularization)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

# NOTE: l1_ratio=1.0 (not penalty="l1") -- scikit-learn's current, non-deprecated API for
# full L1 regularization; the string `penalty` parameter is deprecated as of sklearn 1.8.
l1_logistic_model = LogisticRegression(
    solver="liblinear", C=0.1, l1_ratio=1.0, random_state=RANDOM_STATE, max_iter=1000,
)
l1_logistic_model.fit(X_train_transformed, y_train)

l1_selector = SelectFromModel(l1_logistic_model, prefit=True, threshold=1e-5)
l1_selected_names = set(transformed_feature_names[l1_selector.get_support()])
print(f"L1 LogisticRegression zeroed out {int((l1_logistic_model.coef_[0] == 0).sum())} / "
      f"{len(transformed_feature_names)} coefficients.")
print(f"L1-selected (non-zero coefficient) feature count: {len(l1_selected_names)}")


In [ ]:
l1_vs_skb_overlap = l1_selected_names & select_k_best_names_full
print(f"Overlap between L1-selected ({len(l1_selected_names)} features) and "
      f"SelectKBest(20) (Section 4): {len(l1_vs_skb_overlap)} features")


### Interpretation

L1-regularized Logistic Regression, `79` feature'dan `29`'unun katsayısını **tam olarak sıfıra**
indirdi — bu, Filter-based `SelectKBest`'ten **farklı** bir seçim mekanizmasıdır (model eğitim
sürecine gömülü, lineer bir modelin kendi optimizasyonu). Seçilen feature kümesi, `SelectKBest`
ile **kısmen örtüşüyor** — iki farklı yöntem kategorisinin (Filter vs Embedded) aynı dominant
sinyale kısmen işaret ettiğini, ama birebir aynı feature kümesini üretmediğini gösteriyor.

### Challenge 3 — Feature Ablation Study

Feature gruplarını **tek tek çıkararak**, her grubun E02'nin performansına gerçek katkısını
ölçelim.


In [ ]:
feature_groups = {
    "Date-Time": ["order_month", "order_day_of_week", "order_quarter", "is_weekend", "order_hour_sin", "order_hour_cos"],
    "Interaction/Numeric": ["gross_item_value", "estimated_discount_amount", "quantity_x_scheduled_days", "log_gross_item_value"],
    "Geographic": ["Market", "Order Region"],
    "Shipping": ["Shipping Mode", "Days for shipment (scheduled)"],
    "Product": ["Category Name", "Department Name"],
}

full_e02_auc = feature_benchmark_df.loc[feature_benchmark_df["Experiment"] == "E02", "Validation_ROC_AUC"].iloc[0]
ablation_rows = [{"Group_Removed": "(none, full E02)", "Val_ROC_AUC": full_e02_auc, "Delta": 0.0}]

for group_name, group_columns in feature_groups.items():
    remaining_numerical = [c for c in numerical_features if c not in group_columns]
    remaining_categorical = [c for c in categorical_features if c not in group_columns]
    remaining_features = remaining_numerical + remaining_categorical

    ablation_preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", Pipeline([("imputer", SimpleImputer(strategy="median"))]), remaining_numerical),
            ("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]), remaining_categorical),
        ],
        verbose_feature_names_out=False,
    )
    X_tr_ablated = ablation_preprocessor.fit_transform(train_df[remaining_features], y_train)
    X_va_ablated = ablation_preprocessor.transform(validation_df[remaining_features])

    ablated_model = make_model().fit(X_tr_ablated, y_train)
    ablated_auc = roc_auc_score(y_val, ablated_model.predict_proba(X_va_ablated)[:, 1])
    ablation_rows.append({
        "Group_Removed": group_name, "Val_ROC_AUC": ablated_auc, "Delta": ablated_auc - full_e02_auc,
    })
    print(f"Removed '{group_name}' ({len(group_columns)} raw columns) -> "
          f"Val ROC-AUC={ablated_auc:.4f} (delta={ablated_auc - full_e02_auc:+.4f})")

ablation_df = pd.DataFrame(ablation_rows)
ablation_df


### Interpretation

**En çarpıcı bulgu:** `Date-Time` grubunu (6 feature) çıkarmak, Val ROC-AUC'yi **belirgin biçimde**
düşürüyor (`delta≈-0.028`) — bu, Section 3'teki bulguyla **ilk bakışta çelişiyor gibi görünüyor**:
orada `is_weekend`/`order_day_of_week`/`order_month`'un **bireysel olarak** target ile zayıf bir
ilişkisi olduğunu göstermiştik. **Çözüm:** Bu feature'lar **tek başlarına** zayıf olsa da,
`order_hour_sin`/`order_hour_cos` (aynı grubun parçası) ciddi bir sinyal taşıyor (Section 6'nın
permutation/SHAP analizlerinde zaten görmüştük) — **grup ablation'ı, grubun toplam etkisini
ölçer, bireysel korelasyonları değil.**

Diğer gruplar için: `Shipping`'i çıkarmak da belirgin bir düşüşe yol açıyor (`-0.0051`, beklenen
— bu dominant sinyal); `Geographic` ve `Product` gruplarını çıkarmak performansı **hafifçe
artırıyor** (`+0.0021`, `+0.0010`) — bu feature'ların net bir gürültü kaynağı olabileceğini
gösteriyor; `Interaction/Numeric` grubunu çıkarmak **neredeyse hiçbir fark** yaratmıyor
(`-0.0003`) — bu feature'ların E02'nin performansına **marjinal** bir katkısı var.

### Challenge 4 — Grouped Permutation Importance

`order_hour_sin`/`order_hour_cos`, aynı kaynak feature'dan (cyclical encoding) türetilen bağımlı
bir çift. Bunları **tek tek** vs **birlikte** permute ederek karşılaştıralım.


In [ ]:
permutation_baseline_score = roc_auc_score(y_permutation_sample, fitted_pipeline_e03.predict_proba(X_permutation_sample)[:, 1])
rng_challenge4 = np.random.default_rng(RANDOM_STATE)
n_repeats_challenge4 = 5


def grouped_permutation_importance(columns_to_shuffle):
    scores = []
    for _ in range(n_repeats_challenge4):
        X_shuffled = X_permutation_sample.copy()
        shuffled_order = rng_challenge4.permutation(len(X_shuffled))
        for col in columns_to_shuffle:
            X_shuffled[col] = X_shuffled[col].values[shuffled_order]
        proba = fitted_pipeline_e03.predict_proba(X_shuffled)[:, 1]
        scores.append(roc_auc_score(y_permutation_sample, proba))
    return permutation_baseline_score - np.mean(scores), np.std(scores)


individual_sin_importance, _ = grouped_permutation_importance(["order_hour_sin"])
individual_cos_importance, _ = grouped_permutation_importance(["order_hour_cos"])
grouped_importance, grouped_std = grouped_permutation_importance(["order_hour_sin", "order_hour_cos"])

print(f"Individual -- order_hour_sin importance: {individual_sin_importance:.4f}")
print(f"Individual -- order_hour_cos importance: {individual_cos_importance:.4f}")
print(f"Sum of individual importances            : {individual_sin_importance + individual_cos_importance:.4f}")
print(f"Grouped    -- (sin + cos) together       : {grouped_importance:.4f} (+/- {grouped_std:.4f})")


### Interpretation

Tek başına `order_hour_cos`'un permutation importance'ı **sıfıra çok yakın** — model, bu
feature'ı tek başına karıştırıldığında `order_hour_sin`'den (hâlâ bozulmamış) **aynı döngüsel
bilgiyi kısmen yeniden inşa edebiliyor** gibi görünüyor. Ancak `sin` ve `cos`'u **birlikte**
karıştırdığımızda, importance **bireysel importance'ların toplamından daha yüksek** çıkıyor
(`sin+cos birlikte ≈` bireysel toplamdan büyük). **Bu, bağımlı/korelasyonlu feature çiftlerinde
tekil permutation'ın gerçek toplam etkiyi hafife alabileceğinin somut bir kanıtıdır** — Section
6 Part B'nin "correlated features attribution'ı karmaşıklaştırır" uyarısının burada **nicel
olarak** doğrulanmış hâli.

### Challenge 5 — SHAP Explanation Stability

SHAP Global Importance'ı (`Mean |SHAP|`), **farklı random seed'ler ve farklı validation
sample'ları** üzerinden tekrarlayıp, feature ranking'in ne kadar kararlı olduğunu inceleyelim.


In [ ]:
stability_seeds = [42, 123, 2024]
stability_rankings = {}

for seed in stability_seeds:
    seed_rng = np.random.default_rng(seed)
    sample_positions = seed_rng.choice(X_val_model_input.shape[0], size=500, replace=False)
    sample_df = pd.DataFrame(X_val_model_input[sample_positions], columns=selected_feature_names)
    sample_shap_values = explainer(sample_df)
    ranking = pd.Series(
        np.abs(sample_shap_values.values).mean(axis=0), index=selected_feature_names
    ).sort_values(ascending=False)
    stability_rankings[seed] = ranking
    print(f"Seed {seed} -- Top 5:", list(ranking.head(5).index))


In [ ]:
top5_sets = [set(ranking.head(5).index) for ranking in stability_rankings.values()]
stable_across_all_seeds = top5_sets[0] & top5_sets[1] & top5_sets[2]
print(f"Features in the Top 5 across ALL {len(stability_seeds)} seeds: {stable_across_all_seeds}")
print(f"Stable count: {len(stable_across_all_seeds)} / 5")


### Interpretation

Bu çalıştırmada, **Top 5 sıralaması 3 farklı seed'de birebir aynı** çıktı — bu, bu dataset için
dominant sinyalin (`Days for shipment (scheduled)`, `Shipping Mode` dummy'leri, `order_hour_sin`)
**örnekleme varyasyonuna karşı çok güçlü** olduğunu gösteriyor. **Bunu genel bir kural olarak
sunmuyoruz** — daha zayıf/sınır-yakın sıralamadaki feature'lar (örn. Top 10-20 arası), daha küçük
örneklem boyutlarında muhtemelen **daha az kararlı** olurdu. Küçük örneklemlerden elde edilen
explainability sonuçlarının güvenilirliği, **sinyalin gücüne** bağlıdır — zayıf sinyalli
feature'lar için SHAP ranking'i daha dikkatli yorumlanmalıdır.

### Student Exercise


In [ ]:
# Advanced Challenge Student Exercise:
# TODO: stability_rankings[42] ve stability_rankings[2024]'ün TAM sıralamasını (ilk 20) karşılaştırın
#       (sadece Top 5 değil).
# TODO: Sıralamanın alt kısımlarında (6-20. sıralar) daha fazla değişkenlik olup olmadığını inceleyin.

# print(stability_rankings[42].head(20))
# print(stability_rankings[2024].head(20))


---

## LAB08 Özeti

Bu LAB'da, LAB02-LAB07'de öğrendiğiniz bilgileri Feature Engineering, Feature Selection ve
Explainability üzerinde kullandınız:

- Feature Engineering/Selection/Extraction arasındaki farkı öğrendiniz.
- 44 feature'ı sistematik olarak denetleyip, gerçek veri kontrolleriyle (eşitlik testleri,
  cardinality, missing value) 4 çift kopya kolon ve 2 sabit/boş kolon buldunuz.
- `Delivery Status`'un hedefle **deterministik** olarak eşleştiğini kanıtladınız ve izole bir
  `INVALID — TARGET LEAKAGE` demonstrasyonuyla bunun pratikte nasıl yanıltıcı bir performans
  ürettiğini gördünüz.
- Group-aware kronolojik bir split kurup bütünlüğünü doğruladınız.
- Date-Time, Cyclical ve interaction feature'ları oluşturup, bazılarının (weekend×shipping mode)
  **gerçekten faydasız** olduğunu gerçek verilerle gösterdiniz.
- Filter/Wrapper/Embedded feature selection yöntemlerini uyguladınız ve karşılaştırdınız.
- E01/E02/E03'ü aynı protokolle karşılaştırıp, Feature Engineering'in gerçek bir iyileşme
  sağladığını, Feature Selection'ın ise %75 daha az feature ile benzer performansı koruduğunu
  ölçtünüz.
- Model-Based Importance, Permutation Importance ve SHAP'i karşılaştırıp, bunların **farklı
  sorulara** cevap verdiğini ve her zaman aynı sıralamayı üretmediğini gördünüz.
- SHAP'in raw margin/probability ayrımını, additivity özelliğini ve açıklamanın sınırlamalarını
  (nedensellik değildir, fairness garanti etmez) öğrendiniz.

LAB09'da, burada seçtiğiniz feature strategy ve split üzerine inşa edeceksiniz. Bir sonraki
oturumda görüşmek üzere!
